# LEVELS Benchmark: DSI Data Explorer Agent

This notebook benchmarks the DSI Data Explorer agent across three task-complexity levels and five data-management categories.

## Complexity Levels

| Level | Name | Description | Scoring Method |
|-------|------|-------------|---------|
| **1** | Atomic | Single DSI/API call from natural language | Binary pass/fail: correct tool + task completion |
| **2** | Workflow | Multi-step pipelines (query → filter → summarize, ingest → validate, etc.) | Output correctness + method quality (LLM judge) |
| **3** | Discovery | Hypothesis-level: state a relationship as structured hypothesis (DiscoveryBench-style) | Faceted LLM judge + expert ground truth |

## Data Management Categories

1. Search: external/domain knowledge and literature
2. Data Discovery: locating datasets, files, or records in DSI repositories
3. Data Preparation: cleaning, validating, standardizing scientific data
4. Metadata Generation: structured documentation and provenance
5. Analysis & Summarization: interpreting, aggregating, synthesizing data

These are the DSI tools:

dsi_tools = [

                query_dsi_tool, 

                python_repl_tool, 

                load_dsi_tool, 

                download_file_tool, 

                arxiv_search_tool,

                wikipedia_search_tool,

                web_search_tool,

                send_email_tool,

                upload_paper_tool


## Setup

In [1]:
# Install dependencies into the active Jupyter kernel (run once)
%pip install -q -r requirements.txt
# --no-build-isolation: reuse the setuptools already in this kernel instead of pip
# fetching a fresh copy from PyPI for the build environment (avoids network/SSL
# failures on machines with restricted/proxied access to pypi.org).
%pip install -q --no-build-isolation -e ../..

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Load portal API key from portal_api_key.txt (see portal_api_key.txt.example)
import os
from pathlib import Path

_key_path = Path("portal_api_key.txt")
if not _key_path.is_file():
    raise FileNotFoundError(
        "Missing portal_api_key.txt — copy portal_api_key.txt.example and add your key."
    )
os.environ.setdefault("API_KEY", _key_path.read_text().strip())
os.environ["API_BASE"] = "https://aiportal-api.aws.lanl.gov"

In [3]:
import threading
import contextlib
import io
import json
import re
import subprocess
import sys
import textwrap
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

import pandas as pd
from IPython.display import HTML, Markdown, display
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI

def find_ai_search_dir() -> Path:
    """Locate the tools/ai_search directory from cwd or any parent.

    Walks up the directory tree looking for dsi_explorer_agent.py so the
    notebook still works after the agent changes the process working directory.
    """
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / "dsi_explorer_agent.py").is_file():
            return directory.resolve()
        nested = directory / "tools" / "ai_search" / "dsi_explorer_agent.py"
        if nested.is_file():
            return nested.parent.resolve()
    raise RuntimeError("Could not find tools/ai_search — open this notebook from the dsi repo")

AI_SEARCH_DIR = find_ai_search_dir()
os.chdir(AI_SEARCH_DIR)

from dsi_explorer_agent import DSIExplorer, extract_message_texts
from tool_utils import reset_dsi_call_log, get_dsi_call_log

print(f"Working directory: {AI_SEARCH_DIR}")


Working directory: /Users/hhn/lanl/dsi/tools/ai_search


In [ ]:
# --- Configuration ---
AI_PORTAL_BASE_URL = os.environ.get("API_BASE", "https://aiportal-api.aws.lanl.gov")

POSSIBLE_MODELS = [
    "aws-gov.gpt-5.4",
    "anthropic.claude-sonnet-4-5-20250929-v1:0",
    # "amazon.nova-pro-v1:0",
    # "meta.llama3-70b-instruct-v1:0",
    # "meta.llama3-8b-instruct-v1:0",
    # "mistral7b",
    # "llama3bv32instr",
    # "phi4",
    # "gpt-oss-120b",
    # "e5mistral7b",
    # "NVIDIA-Nemotron-3-Super-120B-A12B-FP8",
    # "gemma-4-31B-it",
]

LLM_MODEL = "aws-gov.gpt-5.4"   # default single model, used by make_explorer()/debug_task() when no llm is passed
JUDGE_MODELS = list(POSSIBLE_MODELS)  # ensemble of judges: every judged score is the AVERAGE across these models,
                                        # so a model under test never sits alone judging its own (or a rival's) transcripts
MODELS_TO_RUN = list(POSSIBLE_MODELS)  # subset this to run the multi-model sweep against fewer models
TASK_TIMEOUT_SEC = 180           # max wall-clock seconds per agent turn before it's aborted as a timeout
USE_MCP = False                 # set True if levels_mcp_server.py is running on :8000
MCP_URL = "http://127.0.0.1:8000/mcp"
MASTER_DB = "data/oceans_11/ocean_11_datasets.db"

# NIF_DB = "data/oceans_11/nif.db"
# POISSON_DB = "data/oceans_11/poisson.db"

RESULTS_DIR = AI_SEARCH_DIR / "levels_benchmark_results"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

def make_llm(model_name: str, temperature: float = 0.1) -> ChatOpenAI:
    """Create a ChatOpenAI client pointed at the LANL AI portal."""
    return ChatOpenAI(
        model=model_name,
        api_key=os.environ["API_KEY"],
        base_url=AI_PORTAL_BASE_URL,
        temperature=temperature,
    )

llm_model = make_llm(LLM_MODEL)
judge_llms = [make_llm(m, temperature=0.0) for m in JUDGE_MODELS]
judge_llm = judge_llms[0]  # back-compat alias for any direct judge_llm.invoke() usage
print(f"Using model: {LLM_MODEL} @ {AI_PORTAL_BASE_URL}")
print(f"Judge ensemble: {JUDGE_MODELS}")


Using model: aws-gov.gpt-5.4 @ https://aiportal-api.aws.lanl.gov
Judge ensemble: ['aws-gov.gpt-5.4', 'anthropic.claude-sonnet-4-5-20250929-v1:0']


In [5]:
def make_explorer(llm: Optional[ChatOpenAI] = None) -> DSIExplorer:
    """Create a fresh DSIExplorer with a new workspace, thread, and master DB loaded.

    Resets cwd to AI_SEARCH_DIR first to avoid nested workspace folders on re-runs.
    Passes MCP settings when USE_MCP is True. Defaults to the global llm_model
    (LLM_MODEL) when no llm is given, so existing single-model calls keep working;
    pass a per-model client (e.g. make_llm(model_name)) to benchmark another model.
    """
    os.chdir(AI_SEARCH_DIR)  # avoid nested workspace folders on re-runs
    return DSIExplorer(
        llm=llm if llm is not None else llm_model,
        db_index_name=MASTER_DB,
        output_mode="jupyter",
        **({"use_mcp": True, "mcp_url": MCP_URL} if USE_MCP else {}),
    )


## Benchmark Harness

In [ ]:
# Ground-truth constants for Level-1 quantitative checks
NIF_DATASET_ID = "3D FLASH Computation of National Ignition Facility Shot"


def canonicalize_db_path(path: str) -> str:
    """Normalize a DB path for exact Level-1 path matching."""
    s = Path(str(path)).as_posix()
    if "oceans_11/" in s:
        return s[s.index("oceans_11/") :]
    return Path(s).name


def _resolve_path_arg(call_args: Dict[str, Any]) -> str:
    """Resolve a load_dsi_tool 'path' arg the same way load_dsi() resolves it internally.

    Both models converged on passing the master/secondary DB's bare filename as `path`
    plus the folder as a separate `master_db_folder` kwarg (load_dsi_tool's own documented
    pattern for resolving relative paths against the master DB's folder) rather than a
    single joined string. Without this, canonicalize_db_path("nif.db") != canonicalize_db_path
    ("oceans_11/nif.db") even though the model loaded the exact right file.
    """
    path = str(call_args.get("path", ""))
    folder = call_args.get("master_db_folder")
    if folder and not Path(path).is_absolute():
        return str(Path(str(folder)) / path)
    return path


def _safe_model_label(model_name: str) -> str:
    """Sanitize a model name (which may contain ':', '.', '/') into a filename-safe label."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", model_name)


@dataclass
class BenchmarkTask:
    """Definition of a single benchmark task (query, rubric, and scoring hints)."""
    task_id: str
    level: int
    category: str
    query: str
    # Level 1 checks
    expected_tools: List[str] = field(default_factory=list)
    expected_args: Dict[str, Any] = field(default_factory=dict)
    expected_path: Optional[str] = None  # canonicalize and compare to tool arg "path"
    expected_count: Optional[int] = None  # must appear in tool result or response
    response_keywords: List[str] = field(default_factory=list)
    forbidden_keywords: List[str] = field(default_factory=list)
    reference_path: List[str] = field(default_factory=list)  # most-efficient human DSI CLI commands, for LLM-judged efficiency scoring
    reference_time_sec: Optional[float] = None  # human wall-clock seconds to execute reference_path manually; RQ1 productivity baseline. Must be measured by a human, not estimated -- left None until that data-collection pass happens.
    setup_query: Optional[str] = None
    timeout_sec: Optional[float] = None  # per-task override; falls back to TASK_TIMEOUT_SEC when None
    steps: List[str] = field(default_factory=list)  # ordered natural-language turns for multi-step Level 2/3 workflows, sent to the SAME explorer so later turns can rely on earlier conversational state. When empty, `query` is sent as a single turn (unchanged Level-1 behavior).
    # Level 2/3 rubric
    success_criteria: str = ""  # free-text rubric judged by score_level2's LLM judge (process-quality/method score)
    ground_truth_hypothesis: Optional[Dict[str, str]] = None  # DiscoveryBench-style facets for score_level3: {"context": ..., "variables": ..., "relationship": ...}
    # Calibration / uncertainty (RQ4): tasks referencing missing data, nonexistent resources, or
    # false/ambiguous premises, where correctly abstaining/asking for clarification IS the pass condition.
    expects_abstention: bool = False
    abstention_keywords: List[str] = field(default_factory=list)  # task-specific hedge phrases; falls back to DEFAULT_ABSTENTION_PHRASES when empty
    requires_network: bool = False
    skip: bool = False


@dataclass
class TaskResult:
    """Outcome of running and scoring one benchmark task."""
    task_id: str
    level: int
    category: str
    query: str
    response: str
    tool_calls: List[Dict[str, Any]]
    elapsed_sec: float
    tokens: int
    passed: Optional[bool] = None
    score: Optional[float] = None
    judge_scores: Optional[Dict[str, Any]] = None
    notes: str = ""
    tool_results: List[str] = field(default_factory=list)
    dsi_calls: List[Dict[str, Any]] = field(default_factory=list)
    model: str = ""  # which LLM produced this result; set by run_benchmark/run_benchmark_all_models
    checkpoint_events: List[Dict[str, Any]] = field(default_factory=list)  # RQ3/RQ7: high-stakes tool calls + hedge language detected post-hoc (metrics only, does not pause execution)
    failure_category: Optional[str] = None  # RQ5: MODEL_REASONING / TOOL_FRAMEWORK / TASK_SPECIFICATION / PROMPT_GAP / ENVIRONMENT_DATA, set by classify_failure() for failed tasks
    failure_rationale: str = ""
    check_results: Optional[Dict[str, bool]] = None  # per-check pass/fail breakdown (e.g. Level 1's tool_ok/args_ok/.../nonempty_ok), set by score_level1() so partial credit is auditable


def extract_tool_calls(result: dict) -> List[Dict[str, Any]]:
    """Collect tool names and args from all AIMessage tool_calls in an agent result."""
    calls = []
    for msg in result.get("messages", []):
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                calls.append({"name": tc.get("name"), "args": tc.get("args", {})})
    return calls


def extract_tool_results(result: dict) -> List[str]:
    """Collect ToolMessage contents as strings for quantitative Level-1 checks."""
    out = []
    for msg in result.get("messages", []):
        if isinstance(msg, ToolMessage):
            content = msg.content
            if isinstance(content, (list, dict)):
                out.append(json.dumps(content, default=str))
            else:
                out.append(str(content))
    return out


def format_dsi_call_log(dsi_calls: List[Dict[str, Any]]) -> str:
    """Pretty-print recorded DSI Python API calls (method, args, kwargs, timing)."""
    if not dsi_calls:
        return "(no DSI API calls)"
    lines = []
    for i, call in enumerate(dsi_calls, 1):
        lines.append(
            f"[{i}] DSI.{call['method']}("
            f"{', '.join(call.get('args', []))}"
            f"{', ' if call.get('args') and call.get('kwargs') else ''}"
            f"{', '.join(f'{k}={v}' for k, v in call.get('kwargs', {}).items())})"
            f"  [{call.get('elapsed_sec', 0):.3f}s]"
            + (f"  ERROR: {call['error']}" if call.get("error") else "")
        )
    return "\n".join(lines)


class TaskTimeoutError(TimeoutError):
    """Raised when a single agent invocation exceeds its allotted timeout."""


def invoke_agent(explorer: DSIExplorer, query: str, timeout: Optional[float] = None) -> dict:
    """Send one user query through the agent graph and return the raw result dict.

    Uses async invoke when MCP is enabled; otherwise sync invoke. Adds _elapsed
    seconds to the result for timing metrics.

    The (blocking) invoke call runs on a daemon thread; if it doesn't finish
    within `timeout` seconds (defaults to TASK_TIMEOUT_SEC), TaskTimeoutError is
    raised so a stuck/slow model can't hang the whole benchmark sweep. Python
    cannot forcibly kill a running thread, so the abandoned call keeps running
    in the background — but because the thread is a daemon, it won't block the
    notebook kernel process from exiting/restarting later, and this function
    itself returns control immediately once `timeout` elapses either way.

    Any stdout/stderr the worker thread writes is captured into a buffer
    instead of going to the real stdout from a background thread. ipykernel
    routes stream output to the currently-executing cell using a parent-header
    that gets confused by concurrent writes from multiple threads; left
    unguarded this silently drops/misattributes *later* print() calls made on
    the main thread (e.g. the "Tools called"/"DSI API calls" lines), which is
    why they tend to stop showing up starting with the second model in a
    multi-model sweep. The buffer is replayed on the main thread right after
    `join()` returns, so only the main thread ever touches the real stdout.
    """
    start = time.time()
    msg = explorer.craft_message(query)
    timeout = TASK_TIMEOUT_SEC if timeout is None else timeout

    box: Dict[str, Any] = {}

    def _invoke():
        buf = io.StringIO()
        try:
            with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                if explorer.use_mcp:
                    from dsi_explorer_agent import run_async
                    box["result"] = run_async(explorer.app.ainvoke(
                        msg, config={"configurable": {"thread_id": explorer.thread_id}}
                    ))
                else:
                    box["result"] = explorer.app.invoke(
                        msg, config={"configurable": {"thread_id": explorer.thread_id}}
                    )
        except Exception as e:
            box["error"] = e
        finally:
            box["captured_output"] = buf.getvalue()

    worker = threading.Thread(target=_invoke, daemon=True)
    worker.start()
    worker.join(timeout=timeout)
    if worker.is_alive():
        raise TaskTimeoutError(f"Agent call exceeded {timeout}s timeout for query: {query[:80]!r}")

    captured = box.get("captured_output", "")
    if captured:
        sys.stdout.write(captured)
        sys.stdout.flush()

    if "error" in box:
        raise box["error"]

    result = box["result"]
    result["_elapsed"] = time.time() - start
    return result


def run_task(explorer: DSIExplorer, task: BenchmarkTask, verbose: bool = True) -> TaskResult:
    """Execute one benchmark task and capture response, tool calls, and timing.

    Runs setup_query first when present (e.g. load a DB before the main query).
    When task.steps is non-empty (Level 2/3 multi-turn workflows), each step is sent in
    order to the SAME explorer, so later steps can rely on earlier conversational state
    (e.g. "which of the two datasets you just examined..."); tool_calls/tool_results/tokens/
    elapsed_sec are accumulated across all steps and the LAST step's reply becomes
    result.response. When task.steps is empty, behaves exactly as before (single-turn
    task.query) -- this is unchanged for every existing Level 1 task.
    Skipped tasks return immediately with notes='skipped'.
    """
    if task.skip:
        return TaskResult(task.task_id, task.level, task.category, task.query,
                          "", [], 0.0, 0, passed=None, notes="skipped")

    reset_dsi_call_log()

    if task.setup_query:
        invoke_agent(explorer, task.setup_query, timeout=task.timeout_sec)

    if verbose:
        display(Markdown(f"### `{task.task_id}` — Level {task.level} / {task.category}\n\n**Query:** {task.query}"))

    turns = task.steps if task.steps else [task.query]
    response = ""
    tool_calls: List[Dict[str, Any]] = []
    tool_results: List[str] = []
    tokens = 0
    elapsed = 0.0
    for i, turn_query in enumerate(turns):
        if verbose and task.steps:
            display(Markdown(f"**Step {i + 1}/{len(turns)}:** {turn_query}"))
        step_result = invoke_agent(explorer, turn_query, timeout=task.timeout_sec)
        response = (step_result.get("response") or "").strip()
        tool_calls.extend(extract_tool_calls(step_result))
        tool_results.extend(extract_tool_results(step_result))
        tokens += step_result.get("metadata", {}).get("token_usage", {}).get("total_tokens", 0)
        elapsed += step_result["_elapsed"]
        if verbose:
            display(Markdown(response))

    dsi_calls = get_dsi_call_log()

    if verbose:
        print(f"Tools called: {[t['name'] for t in tool_calls]} | {elapsed:.1f}s | {tokens} tokens")
        print(f"DSI API calls: {[c['method'] for c in dsi_calls]}")

    return TaskResult(
        task_id=task.task_id,
        level=task.level,
        category=task.category,
        query=task.query,
        response=response,
        tool_calls=tool_calls,
        elapsed_sec=elapsed,
        tokens=tokens,
        tool_results=tool_results,
        dsi_calls=dsi_calls,
    )


def score_level1(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Score a Level 1 (atomic) task with 7 independent binary checks, partial credit.

    Checks: expected tool, expected args (exact), canonicalized path, quantitative
    count in tool results/response, keywords / forbidden keywords, nonempty response.
    Each check defaults to True (vacuously satisfied) when the task doesn't define the
    corresponding field, exactly as before.

    result.check_results stores every check's individual True/False so the breakdown is
    auditable (and displayable per-task, e.g. in l1_df). result.score is now the FRACTION
    of the 7 checks that passed (accumulated partial credit, 0.0-1.0) rather than a strict
    0.0/1.0 binary -- e.g. a task that nails 6 of 7 checks now scores ~0.86 instead of 0.0.
    result.passed remains the strict all-checks-pass flag (unchanged), since pass_rate
    aggregations elsewhere in the notebook rely on it meaning "fully correct".
    """
    tool_names = [t["name"] for t in result.tool_calls]
    matching_calls = [t for t in result.tool_calls if t["name"] in (task.expected_tools or tool_names)]
    notes = []

    tool_ok = True
    if task.expected_tools:
        tool_ok = any(t in tool_names for t in task.expected_tools)
        if not tool_ok:
            notes.append(f"Expected one of {task.expected_tools}, got {tool_names}")

    args_ok = True
    if task.expected_args and tool_ok:
        args_ok = False

        def _arg_match(call_args: Dict[str, Any], key: str, want: Any) -> bool:
            if key == "path":
                return canonicalize_db_path(_resolve_path_arg(call_args)) == canonicalize_db_path(str(want))
            return str(call_args.get(key)) == str(want)

        for call in matching_calls:
            if all(_arg_match(call["args"], k, v) for k, v in task.expected_args.items()):
                args_ok = True
                break
        if not args_ok:
            notes.append(
                f"Expected args {task.expected_args}, got "
                f"{[c['args'] for c in matching_calls]}"
            )

    path_ok = True
    if task.expected_path is not None and tool_ok:
        path_ok = False
        want = canonicalize_db_path(task.expected_path)
        for call in matching_calls:
            got = _resolve_path_arg(call["args"])
            if got and canonicalize_db_path(got) == want:
                path_ok = True
                break
        if not path_ok:
            notes.append(
                f"Expected canonical path {want}, got "
                f"{[c['args'].get('path') for c in matching_calls]}"
            )

    quant_ok = True
    if task.expected_count is not None:
        blob = " ".join(result.tool_results + [result.response])
        # Prefer structured count keys, then bare integer token.
        patterns = [
            rf'["\']?(?:count|missing_count)["\']?\s*[:=]\s*{task.expected_count}\b',
            rf"\b{task.expected_count}\b",
        ]
        quant_ok = any(re.search(p, blob) for p in patterns)
        if not quant_ok:
            notes.append(f"Expected count {task.expected_count} not found in tool results/response")

    keyword_ok = True
    response_lower = result.response.lower()
    tools_blob = " ".join(result.tool_results).lower() + " " + response_lower
    if task.response_keywords:
        keyword_ok = any(kw.lower() in tools_blob for kw in task.response_keywords)
        if not keyword_ok:
            notes.append(f"Response missing keywords: {task.response_keywords}")

    forbidden_ok = True
    if task.forbidden_keywords:
        hits = [kw for kw in task.forbidden_keywords if kw.lower() in tools_blob]
        if hits:
            forbidden_ok = False
            notes.append(f"Forbidden distractors present: {hits}")

    # Atomic tools may return short structured answers; allow short responses when tools ran.
    nonempty = len(result.response.strip()) > 0 or bool(result.tool_calls)
    if not nonempty:
        notes.append("Response too short or empty")

    checks = {
        "tool_ok": tool_ok,
        "args_ok": args_ok,
        "path_ok": path_ok,
        "quant_ok": quant_ok,
        "keyword_ok": keyword_ok,
        "forbidden_ok": forbidden_ok,
        "nonempty_ok": nonempty,
    }
    result.check_results = checks
    result.passed = all(checks.values())
    result.score = round(sum(checks.values()) / len(checks), 3)
    result.notes = "; ".join(notes)
    return result


def llm_judge(prompt: str, llm: Optional[ChatOpenAI] = None) -> dict:
    """Ask a judge LLM to score a task and parse JSON from its response.

    Defaults to the first configured judge model (judge_llm) when `llm` is omitted; pass an
    explicit `llm` (e.g. from judge_llms) to target a specific ensemble member.
    Falls back to {"raw": text} if the model returns non-JSON output.
    """
    resp = (llm or judge_llm).invoke(prompt)
    text = resp.content.strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {"raw": text}


def llm_judge_ensemble(prompt: str) -> List[Dict[str, Any]]:
    """Run the SAME judge prompt against every model in JUDGE_MODELS and return every verdict,
    each tagged with which model produced it ("_judge_model").

    This is what lets every judged score in the harness be an AVERAGE across judge models
    rather than a single model's opinion -- important because JUDGE_MODELS overlaps with
    MODELS_TO_RUN, so without an ensemble a model would sometimes be judging its own (or a
    rival's) transcripts alone. Callers average whichever numeric field(s) they care about
    via _avg_ensemble_field(); for categorical verdicts (see classify_failure), callers check
    for ensemble agreement instead.
    """
    verdicts = []
    for model_name, llm in zip(JUDGE_MODELS, judge_llms):
        verdict = llm_judge(prompt, llm=llm)
        verdict["_judge_model"] = model_name
        verdicts.append(verdict)
    return verdicts


def _avg_ensemble_field(verdicts: List[Dict[str, Any]], key: str) -> Optional[float]:
    """Mean of a numeric field across ensemble verdicts, ignoring judges that omitted it
    (e.g. returned malformed JSON). Returns None if no judge produced a usable value.
    """
    vals = [float(v[key]) for v in verdicts if isinstance(v.get(key), (int, float))]
    return round(sum(vals) / len(vals), 3) if vals else None


def _format_ensemble_reasoning(verdicts: List[Dict[str, Any]], reasoning_key: str = "reasoning") -> str:
    """Concatenate each judge's rationale, prefixed by which model said it, for auditability."""
    return " | ".join(
        f"[{v.get('_judge_model')}] {v.get(reasoning_key, v.get('raw', ''))}" for v in verdicts
    )


def score_efficiency(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """LLM-judge how efficiently the agent solved the task vs. the human reference CLI path.

    Uses task.reference_path (the most-efficient sequence of DSI CLI commands, written by a
    human) as the gold standard, and compares it against the agent's actual tool_calls and
    dsi_calls. The verdict is stored in result.judge_scores so it auto-flattens into the
    results dataframe as judge_efficiency_score / judge_efficiency_reasoning, alongside (and
    independent of) the binary correctness score from score_level1. Tasks with no
    reference_path are skipped (left unscored for efficiency).
    """
    if not task.reference_path:
        return result

    agent_tool_calls = [f"{c['name']}({c['args']})" for c in result.tool_calls]
    agent_dsi_calls = [
        f"{c['method']}({', '.join(c.get('args', []))})" for c in result.dsi_calls
    ]

    prompt = f"""You are judging the EFFICIENCY of an AI agent's path for a data-management task,
against the most efficient path a human expert would take at the DSI command line.

Task: {task.query}

Reference path (ground truth, most efficient -- one command per line):
{chr(10).join(task.reference_path)}

Agent's actual tool calls (in order):
{json.dumps(agent_tool_calls, indent=2)}

Agent's actual underlying DSI API calls (in order):
{json.dumps(agent_dsi_calls, indent=2)}

Score the agent's efficiency from 0.0 (far less efficient than the reference -- redundant
queries, unnecessary exploration, retries, or extra DSI calls beyond what the reference
path needed) to 1.0 (matched the reference path's number of essential steps with no waste).
A correct final answer reached via a wasteful path should still score low here.

Respond with ONLY JSON: {{"efficiency_score": <float 0.0-1.0>, "reasoning": "<one sentence>"}}
"""
    verdicts = llm_judge_ensemble(prompt)
    result.judge_scores = result.judge_scores or {}
    result.judge_scores["efficiency_score"] = _avg_ensemble_field(verdicts, "efficiency_score")
    result.judge_scores["efficiency_score_by_judge"] = {
        v.get("_judge_model"): v.get("efficiency_score") for v in verdicts
    }
    result.judge_scores["efficiency_reasoning"] = _format_ensemble_reasoning(verdicts)
    return result


def score_level2(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Score a Level 2 (workflow) task: tool coverage + quantitative/keyword checks on the
    final response, plus an LLM-judged process-quality score against task.success_criteria.

    Unlike score_level1 (which only requires ANY ONE of expected_tools, since a Level 1
    query has one right tool), a Level 2 workflow must exercise EVERY tool in
    expected_tools at least once -- e.g. a search-then-download-then-summarize task isn't
    "correct" if the agent skipped the download step. Similarly response_keywords are
    ALL-required here (a workflow answer typically needs to report several distinct facts),
    not any-of like Level 1's alternate-phrasing check.

    result.score blends binary correctness (0.5 weight) with the continuous LLM-judged
    process_quality_score (0.5 weight) when success_criteria is set, so a correct-but-
    wasteful or correct-but-poorly-ordered workflow scores lower than a clean one, mirroring
    how score_efficiency already treats Level 1. result.passed reflects correctness only.
    """
    tool_names = [t["name"] for t in result.tool_calls]
    notes = []

    tools_ok = True
    if task.expected_tools:
        missing = [t for t in task.expected_tools if t not in tool_names]
        tools_ok = not missing
        if missing:
            notes.append(f"Missing required tools for this workflow: {missing} (called: {tool_names})")

    blob = " ".join(result.tool_results + [result.response])
    blob_lower = blob.lower()

    quant_ok = True
    if task.expected_count is not None:
        patterns = [
            rf'["\']?(?:count|missing_count)["\']?\s*[:=]\s*{task.expected_count}\b',
            rf"\b{task.expected_count}\b",
        ]
        quant_ok = any(re.search(p, blob) for p in patterns)
        if not quant_ok:
            notes.append(f"Expected count {task.expected_count} not found in tool results/response")

    keyword_ok = True
    if task.response_keywords:
        keyword_ok = all(kw.lower() in blob_lower for kw in task.response_keywords)
        if not keyword_ok:
            missing_kw = [kw for kw in task.response_keywords if kw.lower() not in blob_lower]
            notes.append(f"Response missing required facts/keywords: {missing_kw}")

    forbidden_ok = True
    if task.forbidden_keywords:
        hits = [kw for kw in task.forbidden_keywords if kw.lower() in blob_lower]
        if hits:
            forbidden_ok = False
            notes.append(f"Forbidden distractors present: {hits}")

    nonempty = len(result.response.strip()) > 0
    if not nonempty:
        notes.append("Response too short or empty")

    correctness_ok = all([tools_ok, quant_ok, keyword_ok, forbidden_ok, nonempty])

    process_score = None
    if task.success_criteria:
        prompt = f"""You are judging the METHOD QUALITY of an AI agent's multi-step workflow for a
data-management task, against a rubric describing what a good approach looks like.

Task: {task.query}

Rubric (what a high-quality workflow should do): {task.success_criteria}

Agent's tool calls (in order): {json.dumps([f"{c['name']}({c['args']})" for c in result.tool_calls], indent=2)}

Agent's final response: {result.response}

Score process quality from 0.0 (ignored the rubric, wrong/haphazard approach) to 1.0 (followed
a sound, efficient multi-step approach matching the rubric's intent).

Respond with ONLY JSON: {{"process_quality_score": <float 0.0-1.0>, "reasoning": "<one sentence>"}}
"""
        verdicts = llm_judge_ensemble(prompt)
        process_score = _avg_ensemble_field(verdicts, "process_quality_score")
        result.judge_scores = result.judge_scores or {}
        result.judge_scores["process_quality_score"] = process_score
        result.judge_scores["process_quality_score_by_judge"] = {
            v.get("_judge_model"): v.get("process_quality_score") for v in verdicts
        }
        result.judge_scores["process_quality_reasoning"] = _format_ensemble_reasoning(verdicts)

    result.passed = correctness_ok
    correctness_component = 1.0 if correctness_ok else 0.0
    if isinstance(process_score, (int, float)):
        result.score = round(0.5 * correctness_component + 0.5 * float(process_score), 3)
    else:
        result.score = correctness_component
    result.notes = "; ".join(notes)
    return result


def score_level3(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Score a Level 3 (discovery/hypothesis) task with a DiscoveryBench-style faceted judge.

    Compares the agent's final response against task.ground_truth_hypothesis facets
    (context, variables, relationship) written by a human who examined the same data, each
    scored 0.0-1.0 by the judge LLM. result.score is the mean facet score; result.passed uses
    a 0.6 threshold rather than Level 1/2's stricter binary checks, since partial credit is
    the norm for open-ended discovery (the agent may find the right relationship but describe
    the wrong variables, etc.) -- see judge_scores/notes for which facet(s) were missed.
    """
    gt = task.ground_truth_hypothesis or {}
    if not gt:
        result.notes = "no ground_truth_hypothesis defined on this task; skipping level-3 scoring"
        result.score = None
        result.passed = None
        return result

    prompt = f"""You are judging a scientific HYPOTHESIS an AI agent proposed after analyzing a DSI
scientific dataset, against a ground-truth hypothesis written by a human expert who examined the
same data (DiscoveryBench-style faceted scoring). Score each facet from 0.0 (not addressed or
contradicted) to 1.0 (fully matches in substance, wording may differ).

Task: {task.query}

Ground-truth hypothesis facets:
- context: {gt.get('context', '')}
- variables: {gt.get('variables', '')}
- relationship: {gt.get('relationship', '')}

Agent's final response:
{result.response}

Respond with ONLY JSON: {{"context_score": <0.0-1.0>, "variables_score": <0.0-1.0>,
"relationship_score": <0.0-1.0>, "reasoning": "<1-2 sentences noting any facet missed or contradicted>"}}
"""
    verdicts = llm_judge_ensemble(prompt)
    context_score = _avg_ensemble_field(verdicts, "context_score")
    variables_score = _avg_ensemble_field(verdicts, "variables_score")
    relationship_score = _avg_ensemble_field(verdicts, "relationship_score")
    facet_scores = [s for s in (context_score, variables_score, relationship_score) if isinstance(s, (int, float))]
    hypothesis_reasoning = _format_ensemble_reasoning(verdicts)

    result.judge_scores = result.judge_scores or {}
    result.judge_scores.update({
        "context_score": context_score,
        "variables_score": variables_score,
        "relationship_score": relationship_score,
        "hypothesis_reasoning": hypothesis_reasoning,
    })
    result.score = round(sum(facet_scores) / len(facet_scores), 3) if facet_scores else 0.0
    result.passed = result.score >= 0.6
    result.notes = hypothesis_reasoning
    return result


# RQ4 (does the agent know what it doesn't know?): fallback hedge/abstention phrases used by
# score_calibration() when a task doesn't define its own abstention_keywords.
DEFAULT_ABSTENTION_PHRASES = [
    "not found", "no such", "does not exist", "doesn't exist", "cannot find", "can't find",
    "could not find", "couldn't find", "unable to determine", "unable to locate",
    "no information", "not available", "not specified", "no matching", "no records",
    "none found", "no datasets", "not documented", "could you clarify", "could you specify",
    "which dataset", "can you clarify", "not sure", "i don't know", "i do not know",
    "no such column", "no such field", "unclear", "does not specify", "not provided",
]


def score_calibration(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Score a calibration ("trap") task (RQ4): the task references missing data, a
    nonexistent resource, or a false/ambiguous premise, and the CORRECT behavior is to
    abstain or ask for clarification rather than confidently fabricate an answer.

    Passes (score=1.0) if the response contains abstention/clarification language
    (task.abstention_keywords, falling back to DEFAULT_ABSTENTION_PHRASES); otherwise fails
    (score=0.0) and result.notes is tagged "HALLUCINATION" so it's easy to filter for in
    analysis. This is a cheap keyword heuristic, not an LLM judge -- it will have false
    negatives (an agent that hedges in an unanticipated phrasing) but very few false
    positives, which is the safer failure mode for measuring overconfidence.
    """
    phrases = [p.lower() for p in (task.abstention_keywords or DEFAULT_ABSTENTION_PHRASES)]
    response_lower = result.response.lower()
    abstained = any(p in response_lower for p in phrases)

    result.passed = abstained
    result.score = 1.0 if abstained else 0.0
    result.notes = (
        "correctly abstained / flagged uncertainty" if abstained
        else "HALLUCINATION: answered confidently despite missing, nonexistent, or ambiguous data"
    )
    return result


# RQ3/RQ7 (autonomy & human-guidance): tools whose effects are hard to undo (writes data to
# disk, sends external email, uploads to a third-party API) -- a real human-in-the-loop system
# would plausibly gate these behind an approval step.
HIGH_STAKES_TOOLS = {"download_file_tool", "send_email_tool", "upload_paper_tool"}
# Hedging language in the agent's own final response -- a signal the agent itself was unsure.
HEDGE_PHRASES = [
    "i think", "probably", "not certain", "not sure", "may be", "might be", "possibly",
    "i believe", "seems like", "appears to be", "could be wrong", "unclear", "i'm not 100%",
]


def detect_checkpoint_opportunities(result: TaskResult) -> TaskResult:
    """Post-hoc, METRICS-ONLY detector for where a human-in-the-loop checkpoint would have
    fired had one existed (RQ3/RQ7). This does not pause, gate, or alter execution in any
    way -- it only annotates result.checkpoint_events so later analysis can ask "how often
    would a human have been consulted, and did that correlate with pass/fail?" as an
    observational first pass. See BENCHMARK_METHODOLOGY.md for the proposed follow-on study
    that actually runs the same tasks under simulated autonomy tiers.

    Flags two kinds of events per task: (a) any call to a HIGH_STAKES_TOOLS tool (an
    irreversible/external-effect action), (b) HEDGE_PHRASES found in the final response.
    """
    events = []
    for call in result.tool_calls:
        if call.get("name") in HIGH_STAKES_TOOLS:
            events.append({"type": "high_stakes_action", "tool": call["name"], "args": call.get("args", {})})
    response_lower = result.response.lower()
    for phrase in HEDGE_PHRASES:
        if phrase in response_lower:
            events.append({"type": "hedge_language", "phrase": phrase})
    result.checkpoint_events = events
    return result


# RQ5 (when the agent fails, why?): failure-attribution taxonomy.
FAILURE_CATEGORIES = [
    "MODEL_REASONING",    # tools/data were available; the model chose poorly or misread a result
    "TOOL_FRAMEWORK",     # a tool exception, agent-framework bug, or timeout occurred
    "TASK_SPECIFICATION", # the task/rubric is ambiguous or the scorer is too strict for a reasonable answer
    "PROMPT_GAP",         # the agent's system prompt (dsi_explorer_agent.py) lacked guidance that would have helped
    "ENVIRONMENT_DATA",   # network/API/file unavailability outside the agent's control
]


def classify_failure(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """LLM-judge the most likely ROOT CAUSE of a failed task (RQ5), so failures can be
    attributed to the model, the agent framework, the task/rubric design, the system
    prompt, or the environment instead of being lumped into one pass/fail bucket.

    Only meaningful on a TaskResult that already failed (result.passed is False); left
    untouched otherwise. Feeds the judge LLM the query, the scorer's own notes (which
    already often hint at the cause, e.g. "timeout:" or "error:"), the agent's tool calls,
    and its final response, and asks for exactly one category from FAILURE_CATEGORIES plus
    a one-sentence rationale, stored on result.failure_category / result.failure_rationale.

    Unlike the numeric judges (efficiency/process-quality/hypothesis), a category label can't
    be "averaged" across the judge ensemble -- instead this uses AGREEMENT: if every judge in
    JUDGE_MODELS picks the same category, that's the result; if they disagree, the category is
    "UNKNOWN" rather than arbitrarily picking one judge's opinion, and every judge's individual
    pick + rationale is preserved in result.failure_rationale for manual review.
    """
    if result.passed:
        return result

    prompt = f"""An AI agent failed a data-management benchmark task. Classify the MOST LIKELY
root cause into exactly one of these categories:

- MODEL_REASONING: tools/data were available; the model picked the wrong approach or misread a result
- TOOL_FRAMEWORK: a tool exception, agent-framework bug, or timeout occurred
- TASK_SPECIFICATION: the task or its scoring rubric is ambiguous / overly strict for a reasonable answer
- PROMPT_GAP: the agent's system prompt lacked guidance that would have prevented this
- ENVIRONMENT_DATA: network, external API, or file unavailability outside the agent's control

Task query: {task.query}
Scorer notes: {result.notes}
Agent's tool calls: {json.dumps(result.tool_calls, default=str)[:2000]}
Agent's final response: {result.response[:1000]}

Respond with ONLY JSON: {{"failure_category": "<one of the five category names above, exactly as spelled>", "rationale": "<one sentence>"}}
"""
    verdicts = llm_judge_ensemble(prompt)
    categories = [v.get("failure_category") for v in verdicts]
    valid_categories = [c for c in categories if c in FAILURE_CATEGORIES]
    if valid_categories and len(set(valid_categories)) == 1:
        result.failure_category = valid_categories[0]
    else:
        result.failure_category = "UNKNOWN"
    result.failure_rationale = " | ".join(
        f"[{v.get('_judge_model')}] {v.get('failure_category')}: {v.get('rationale', v.get('raw', ''))}"
        for v in verdicts
    )
    return result


def score_task(task: BenchmarkTask, result: TaskResult) -> TaskResult:
    """Route a TaskResult to the scorer for its complexity level, then layer on efficiency,
    checkpoint-opportunity detection, and (for failures) root-cause classification.

    Calibration ("trap") tasks are routed by task.expects_abstention regardless of level,
    since the pass condition there (abstain/ask for clarification) is orthogonal to the
    level 1/2/3 correctness rubrics.
    """
    if task.expects_abstention:
        result = score_calibration(task, result)
    elif task.level == 1:
        result = score_level1(task, result)
    elif task.level == 2:
        result = score_level2(task, result)
    else:
        result = score_level3(task, result)
    result = score_efficiency(task, result)
    result = detect_checkpoint_opportunities(result)
    result = classify_failure(task, result)
    return result


def run_benchmark(
    tasks: List[BenchmarkTask],
    *,
    isolated: bool = False,
    llm: Optional[ChatOpenAI] = None,
    model_label: str = "",
) -> List[TaskResult]:
    """Run and score a list of tasks, printing PASS/FAIL/SKIP for each.

    When isolated=True (Level 1), each task gets a fresh explorer so prior
    conversation state does not affect the next task. When isolated=False,
    all tasks share one explorer (useful for Level 2/3 workflow continuity).

    Pass `llm` (e.g. make_llm(model_name)) to benchmark a specific model instead
    of the global default; `model_label` is stamped onto every TaskResult so
    results from different models can be told apart and compared afterward.

    Tasks with requires_network are skipped when LEVELS_SKIP_NETWORK=1. Any task
    that raises is caught and recorded as a failed TaskResult instead of aborting
    the rest of the sweep — this includes TaskTimeoutError when a task exceeds
    its timeout_sec / TASK_TIMEOUT_SEC, and other errors (e.g. a model that
    errors on tool-calling).
    """
    results = []
    explorer = None if isolated else make_explorer(llm=llm)
    for task in tasks:
        if task.requires_network and os.environ.get("LEVELS_SKIP_NETWORK", "0") == "1":
            task = BenchmarkTask(**{**asdict(task), "skip": True})
        task_explorer = make_explorer(llm=llm) if isolated else explorer
        task_timeout = task.timeout_sec if task.timeout_sec is not None else TASK_TIMEOUT_SEC
        try:
            tr = run_task(task_explorer, task)
            tr = score_task(task, tr)
        except TaskTimeoutError as e:
            tr = TaskResult(
                task_id=task.task_id, level=task.level, category=task.category, query=task.query,
                response="", tool_calls=[], elapsed_sec=task_timeout, tokens=0,
                passed=False, score=0.0, notes=f"timeout: {e}",
            )
        except Exception as e:
            tr = TaskResult(
                task_id=task.task_id, level=task.level, category=task.category, query=task.query,
                response="", tool_calls=[], elapsed_sec=0.0, tokens=0,
                passed=False, score=0.0, notes=f"error: {e!r}",
            )
        tr.model = model_label
        results.append(tr)
        prefix = f"{model_label} | " if model_label else ""
        if tr.notes == "skipped":
            status = "SKIP"
        elif tr.notes.startswith("timeout:"):
            status = "TIMEOUT"
        elif tr.passed:
            status = "PASS"
        else:
            status = "FAIL"
        print(f"[{status}] {prefix}{tr.task_id} — score={tr.score}\n")
    return results


def results_to_dataframe(results: List[TaskResult], tasks: Optional[List[BenchmarkTask]] = None) -> pd.DataFrame:
    """Flatten TaskResult objects into a summary DataFrame for display and analysis.

    Pass the corresponding `tasks` list to additionally compute RQ1 productivity columns
    (time_ratio / effective_productivity) for any task with reference_time_sec set.
    """
    task_by_id = {t.task_id: t for t in (tasks or [])}
    rows = []
    for r in results:
        row = {
            "model": r.model,
            "task_id": r.task_id,
            "query": task_by_id[r.task_id].query if r.task_id in task_by_id else "",
            "level": r.level,
            "category": r.category,
            "passed": r.passed,
            "score": r.score,
            "elapsed_sec": round(r.elapsed_sec, 1),
            "tokens": r.tokens,
            "tools": ", ".join(t["name"] for t in r.tool_calls),
            "response": r.response,
            "notes": r.notes,
            "expects_abstention": task_by_id[r.task_id].expects_abstention if r.task_id in task_by_id else None,
            "hallucination_detected": r.notes.startswith("HALLUCINATION") if r.notes else False,
            "checkpoint_opportunity_count": len(r.checkpoint_events),
            "high_stakes_action_count": sum(1 for e in r.checkpoint_events if e.get("type") == "high_stakes_action"),
            "hedge_language_count": sum(1 for e in r.checkpoint_events if e.get("type") == "hedge_language"),
            "failure_category": r.failure_category,
        }
        task = task_by_id.get(r.task_id)
        if task is not None and task.reference_time_sec:
            row["reference_time_sec"] = task.reference_time_sec
            row["time_ratio"] = round(r.elapsed_sec / task.reference_time_sec, 2)
            row["productivity_gain"] = round(1 - row["time_ratio"], 2)
        if r.check_results:
            for k, v in r.check_results.items():
                row[f"check_{k}"] = v
        if r.judge_scores:
            for k, v in r.judge_scores.items():
                if isinstance(v, (int, float, str, bool)):
                    row[f"judge_{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)


def effective_productivity(df: pd.DataFrame) -> pd.DataFrame:
    """RQ1: 'did agentic assistance improve productivity?' A task only counts as a
    productivity win if the agent BOTH passed it AND finished faster than the human
    reference_time_sec baseline -- a fast wrong answer doesn't count. Requires
    reference_time_sec to have been populated on the tasks (see BenchmarkTask docstring);
    returns an empty frame with a warning if no task has that field set yet.
    """
    if "time_ratio" not in df.columns or df["time_ratio"].dropna().empty:
        print(
            "No reference_time_sec set on any task yet -- effective_productivity requires a "
            "human to time execution of each task's reference_path once. See "
            "BENCHMARK_METHODOLOGY.md (RQ1) for the data-collection protocol."
        )
        return pd.DataFrame()
    sub = df.dropna(subset=["time_ratio"]).copy()
    return (
        sub.groupby("model")
        .apply(lambda g: pd.Series({
            "pass_rate": g["passed"].mean(),
            "mean_productivity_gain_when_passed": g.loc[g["passed"] == True, "productivity_gain"].mean(),
            "effective_productivity": g["passed"].mean() * g.loc[g["passed"] == True, "productivity_gain"].mean(),
            "n_tasks": len(g),
        }))
        .round(3)
    )


def _current_git_commit() -> str:
    """Best-effort short git commit hash for reproducibility metadata; '' if unavailable."""
    try:
        out = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"], cwd=str(AI_SEARCH_DIR),
            capture_output=True, text=True, timeout=5,
        )
        return out.stdout.strip() if out.returncode == 0 else ""
    except Exception:
        return ""


def save_results(results: List[TaskResult], label: str) -> Path:
    """Write benchmark results to levels_benchmark_results/{RUN_ID}_{label}.json, plus a
    sibling {RUN_ID}_{label}.meta.json with run-level reproducibility metadata (git commit,
    timeout/MCP config) that isn't repeated per-task -- addresses the "no run metadata"
    reproducibility gap called out in BENCHMARK_METHODOLOGY.md.
    """
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    out = RESULTS_DIR / f"{RUN_ID}_{label}.json"
    payload = [asdict(r) for r in results]
    out.write_text(json.dumps(payload, indent=2, default=str))

    meta_out = RESULTS_DIR / f"{RUN_ID}_{label}.meta.json"
    meta = {
        "run_id": RUN_ID,
        "label": label,
        "git_commit": _current_git_commit(),
        "task_timeout_sec": TASK_TIMEOUT_SEC,
        "use_mcp": USE_MCP,
        "master_db": MASTER_DB,
        "n_tasks": len(results),
    }
    meta_out.write_text(json.dumps(meta, indent=2, default=str))
    print(f"Saved: {out}")
    return out

def format_dsi_call_log(dsi_calls: List[Dict[str, Any]]) -> str:
    """Pretty-print recorded DSI Python API calls (method, args, kwargs, timing)."""
    if not dsi_calls:
        return "(no DSI API calls)"
    lines = []
    for i, call in enumerate(dsi_calls, 1):
        lines.append(
            f"[{i}] DSI.{call['method']}("
            f"{', '.join(call.get('args', []))}"
            f"{', ' if call.get('args') and call.get('kwargs') else ''}"
            f"{', '.join(f'{k}={v}' for k, v in call.get('kwargs', {}).items())})"
            f"  [{call.get('elapsed_sec', 0):.3f}s]"
            + (f"  ERROR: {call['error']}" if call.get("error") else "")
        )
    return "\n".join(lines)


## Task Catalog

Tasks are drawn from the example queries in `dsi_data_explorer.ipynb`, `dsi_data_explorer-poisson.ipynb`, and `dsi_code_run.ipynb`.

In [7]:
# Level 1 — Atomic (single tool call, binary pass/fail + quantitative checks)
# Load data, store data, query data, extract data, search data, download data
#
# Each task's `reference_path` is the most-efficient sequence of DSI CLI commands a human
# would run to answer the same query. score_efficiency() LLM-judges the agent's actual
# tool_calls/dsi_calls against this reference to produce a judge_efficiency_score,
# independent of the binary correctness score from score_level1.

LEVEL1_TASKS = [
    BenchmarkTask(
        task_id="L1_load_ocean11",
        level=1, category="Data Access",
        query="Load the ocean_11_datasets.db database.",
        expected_tools=["load_dsi_tool"],
        expected_path= "oceans_11/ocean_11_datasets.db",
        expected_args={"path": "oceans_11/ocean_11_datasets.db"},
        reference_path=["read /Users/hhn/lanl/dsi/tools/ai_search/oceans11.db"],
    ),
    BenchmarkTask(
        task_id="L1_load_nif_direct",
        level=1, category="Data Access",
        query="Load the nif.db database.",
        expected_tools=["load_dsi_tool"],
        expected_path="oceans_11/nif.db",
        expected_args={"path": "oceans_11/nif.db"},
        reference_path=["read data/oceans_11/nif.db"],
    ),
    BenchmarkTask(
        task_id="L1_load_poisson",
        level=1, category="Data Access",
        query="Load the poisson.db database.",
        expected_tools=["load_dsi_tool"],
        expected_path="oceans_11/poisson.db",
        expected_args={"path": "oceans_11/poisson.db"},
        reference_path=["read data/oceans_11/poisson.db"],
    ),
    BenchmarkTask(
        task_id="L1_search_implosion",
        level=1, category="Data Discovery",
        query="Do you have any implosion data in oceans 11 database?",
        expected_tools=["query_dsi_tool"],
        response_keywords=["National Ignition", "NIF", "FLASH"],
        forbidden_keywords=["Bowtie", "Deep Water Impact"],
        reference_path=[
            "read /Users/hhn/lanl/dsi/tools/ai_search/oceans11.db",
            "search oceans11 implosion",
        ],
    ),
    BenchmarkTask(
        task_id="L1_download_nif_sim",
        level=1, category="Data Retrieval",
        query="Download the simulation data for sim name: nifcylxyz_hdf5_plt_cnt_0000.flash",
        expected_tools=["download_file_tool"],
        expected_args={"url": "https://oceans11.lanl.gov/nif/N210201-001_3D_r2//nifcylxyz_hdf5_plt_cnt_0000.flash"},
        requires_network=True,
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, dsi_database FROM genesis_datacard WHERE Title LIKE '%NIF%' OR Title LIKE '%FLASH%'\"",
            "read data/oceans_11/nif.db",
            "query \"SELECT sim_name, link FROM nif_metadata WHERE sim_name LIKE '%nifcylxyz_hdf5_plt_cnt_0000%'\"",
            "pull_data url nif_sim https://oceans11.lanl.gov/nif/N210201-001_3D_r2//nifcylxyz_hdf5_plt_cnt_0000.flash",
        ],
    ),
    BenchmarkTask(
        task_id="L1_store_poisson_hdf5",
        level=1, category="Data Retrieval",
        query="Download this simulation data file: https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5",
        expected_tools=["download_file_tool"],
        expected_args={"url": "https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5"},
        requires_network=True,
        reference_path=[
            "pull_data url poisson_hdf5 https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5",
        ],
    ),
    BenchmarkTask(
        task_id="L1_owner_field",
        level=1, category="Metadata Query",
        query=f'Who is the owner of the NIF dataset"',
        expected_tools=["query_dsi_tool"],
        response_keywords=["Joshua", "Paul", "Sauppe"],
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, Contact_Point_, Email FROM genesis_datacard WHERE Title LIKE '%NIF%' OR Title LIKE '%FLASH%'\"",
        ],
    ),
    BenchmarkTask(
        task_id="L1_extract_dataset_count",
        level=1, category="Data Exploration",
        query="How many datasets are in the Ocean 11 catalog?",
        expected_tools=["query_dsi_tool"],
        expected_count=9,
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            'query "SELECT COUNT(*) FROM genesis_datacard"',
        ],
    ),
    BenchmarkTask(
        task_id="L1_extract_originating_org",
        level=1, category="Metadata Query",
        query="What organization is listed for the Ocean 11 datasets?",
        expected_tools=["query_dsi_tool"],
        response_keywords=["DOE", "LANL"],
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            'query "SELECT DISTINCT Originating_Organization FROM genesis_datacard"',
        ],
    ),
    BenchmarkTask(
        task_id="L1_extract_physics_theme_count",
        level=1, category="Metadata Query",
        query="How many datasets are categorized as physics?",
        expected_tools=["query_dsi_tool"],
        expected_count=5,
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT COUNT(*) FROM genesis_datacard WHERE LOWER(\\\"Theme/Category_:_Domain\\\") = 'physics'\"",
        ],
    ),
    BenchmarkTask(
        task_id="L1_nif_domain_extent",
        level=1, category="Data Exploration",
        query="What are the spatial bounds of the NIF simulation's computational domain?",
        expected_tools=["query_dsi_tool"],
        response_keywords=["3.072"],
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "search computational domain",
        ],
    ),
    BenchmarkTask(
        task_id="L1_nif_timestep_count",
        level=1, category="Data Exploration",
        query="How many discrete timesteps of the NIF simulation are recorded in NIF.",
        expected_tools=["query_dsi_tool"],
        expected_count=20,
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT COUNT(DISTINCT timestep) FROM nif_metadata"',
        ],
    ),
    BenchmarkTask(
        task_id="L1_nif_array_field_count",
        level=1, category="Data Exploration",
        query="How many distinct physical quantity arrays are recorded per simulation timestep in NIF.",
        expected_tools=["query_dsi_tool"],
        expected_count=9,
        response_keywords=["dens", "pres"],
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT * FROM array_and_types"',
        ],
    ),
    # cd tools/ai_search
    # dsi
    # dsi> search dens
    # dsi> search dens_max *returns no results*
    # Too complicated for level 1
    # BenchmarkTask(
    #     task_id="L1_dens_max_units",
    #     level=1, category="Metadata Generation",
    #     query="What unit is dens_max recorded in?",
    #     expected_tools=["query_dsi_tool"],
    #     response_keywords=["g/cm"],
    # ),

]


### Level 2 — Workflow Tasks

Multi-step pipelines combining ≥2 data-management categories (e.g. search → download →
summarize, filter → report, ingest → validate). Run **non-isolated** (see `run_level`), so
tasks sharing a list may see each other's conversational context; tasks that need guaranteed
clean state use `setup_query` or an explicit `steps` list of turns run against one shared
explorer (`task.steps`, see `run_task`). Scored by `score_level2()`: every tool in
`expected_tools` must be used at least once (not any-of, like Level 1), plus an LLM-judged
process-quality score against `success_criteria`. Ground-truth counts/fields below were
hand-verified against `data/oceans_11/*.db` directly with sqlite3, the same way Level 1's
were verified via the DSI CLI.

In [8]:
LEVEL2_TASKS = [
    BenchmarkTask(
        task_id="L2_search_download_summarize",
        level=2, category="Analysis & Summarization",
        query=(
            "Find the NIF simulation dataset in the Ocean 11 catalog, download the file for "
            "sim name nifcylxyz_hdf5_plt_cnt_0000.flash, and then report who owns the dataset "
            "and how many discrete timesteps the simulation has."
        ),
        expected_tools=["query_dsi_tool", "download_file_tool"],
        response_keywords=["Sauppe", "20"],
        requires_network=True,
        success_criteria=(
            "Should query the Ocean 11 catalog to find the NIF dataset's DB link, download the "
            "specified simulation file, and query nif.db to report both the owner's name "
            "(Joshua Paul Sauppe) and the timestep count (20) without redundant re-exploration."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, Contact_Point_, dsi_database FROM genesis_datacard WHERE Title LIKE '%NIF%' OR Title LIKE '%FLASH%'\"",
            "read data/oceans_11/nif.db",
            "query \"SELECT sim_name, link FROM nif_metadata WHERE sim_name LIKE '%nifcylxyz_hdf5_plt_cnt_0000%'\"",
            "pull_data url nif_sim https://oceans11.lanl.gov/nif/N210201-001_3D_r2//nifcylxyz_hdf5_plt_cnt_0000.flash",
            'query "SELECT COUNT(DISTINCT timestep) FROM nif_metadata"',
        ],
    ),
    BenchmarkTask(
        task_id="L2_physics_contact_report",
        level=2, category="Metadata Generation",
        query=(
            "List the titles of every physics-themed dataset in the Ocean 11 catalog "
            "(case-insensitive) along with each one's point of contact, and tell me which "
            "contact person appears most often among them."
        ),
        expected_tools=["query_dsi_tool"],
        expected_count=5,
        response_keywords=["Nathan"],
        success_criteria=(
            "Should identify all 5 physics-themed datasets (case-insensitive match on the theme "
            "field: Deep Water Impact, Higrad Firetex Wildfire, Gray-Scott reaction-diffusion, "
            "Heat Equations, and Poisson's Equations for Electrostatics), list their contacts, "
            "and correctly identify that Nathan DeBardeleben (or a near-identical spelling "
            "variant on the Poisson record) is associated with the most of them (3 of 5: "
            "Gray-Scott, Heat Equations, and Poisson)."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Title, Contact_Point_ FROM genesis_datacard WHERE LOWER(\\\"Theme/Category_:_Domain\\\") = 'physics'\"",
        ],
    ),
    BenchmarkTask(
        task_id="L2_poisson_field_completeness",
        level=2, category="Data Preparation",
        query=(
            "Load the Poisson database and check whether its single record has a value in "
            "every field. Report the number of empty, null, or missing fields you find."
        ),
        expected_tools=["load_dsi_tool", "query_dsi_tool"],
        expected_count=0,
        success_criteria=(
            "Should load poisson.db, inspect the single record across all 7 fields (name, LAUR, "
            "example, environment, argements, data, statistics), and correctly report that none "
            "are missing/empty/null -- all seven contain populated URLs."
        ),
        reference_path=[
            "read data/oceans_11/poisson.db",
            'query "SELECT * FROM poisson"',
        ],
    ),
    BenchmarkTask(
        task_id="L2_cross_db_record_compare",
        level=2, category="Analysis & Summarization",
        query="Compare the number of discrete records in the NIF dataset vs. the Poisson dataset.",
        steps=[
            "Load nif.db and tell me how many discrete timesteps are recorded.",
            "Now load poisson.db and tell me how many records it contains.",
            "Of the two datasets you just examined, which has more discrete records, and by how many?",
        ],
        expected_tools=["load_dsi_tool", "query_dsi_tool"],
        expected_count=19,
        response_keywords=["nif", "poisson"],
        success_criteria=(
            "Should recall (or re-verify) NIF's 20 timesteps and Poisson's 1 record from earlier "
            "in this same conversation and correctly conclude NIF has 19 more discrete records "
            "than Poisson, ideally without wastefully re-loading/re-querying data it already "
            "has the answer for."
        ),
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT COUNT(DISTINCT timestep) FROM nif_metadata"',
            "read data/oceans_11/poisson.db",
            'query "SELECT COUNT(*) FROM poisson"',
        ],
    ),
    BenchmarkTask(
        task_id="L2_dsi_database_link_audit",
        level=2, category="Metadata Generation",
        query=(
            "For every dataset in the Ocean 11 catalog, check whether it has a linked DSI "
            "sub-database (the dsi_database field) and report how many of the 9 datasets are "
            "missing one."
        ),
        expected_tools=["query_dsi_tool"],
        expected_count=7,
        success_criteria=(
            "Should query the dsi_database field across all 9 genesis_datacard records and "
            "correctly report that 7 of the 9 datasets have no linked DSI sub-database (only "
            "the NIF and Poisson entries do)."
        ),
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            'query "SELECT Title, dsi_database FROM genesis_datacard"',
        ],
    ),
    BenchmarkTask(
        task_id="L2_poisson_args_filename_facts",
        level=2, category="Data Preparation",
        query=(
            "Download the Poisson dataset's simulation-arguments file (the 'argements' field "
            "in the Poisson catalog record) and, from its filename or contents, report the "
            "grid size and number of samples in this dataset."
        ),
        expected_tools=["query_dsi_tool", "download_file_tool"],
        response_keywords=["32x32", "1000"],
        requires_network=True,
        success_criteria=(
            "Should query poisson.db for the 'argements' URL, download it, and correctly "
            "extract '32x32' as the grid size and '1000' (from the 1-1000 sample range in the "
            "filename/contents) as the sample count, without fabricating different numbers."
        ),
        reference_path=[
            "read data/oceans_11/poisson.db",
            'query "SELECT argements FROM poisson"',
            "pull_data url poisson_args https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/arguments_electrostatic_poisson_32x32_1-1000.json",
        ],
    ),
]

### Level 3 — Discovery Tasks

Hypothesis-level, DiscoveryBench-style tasks: the agent must analyze real data and state a
finding as a structured hypothesis, scored against `ground_truth_hypothesis` facets
(`context`/`variables`/`relationship`) by a faceted LLM judge (`score_level3`). Every ground
truth below was verified directly against `data/oceans_11/*.db` with sqlite3 (see chat
history / `BENCHMARK_METHODOLOGY.md` for the verification queries) rather than assumed, since
a wrong ground truth would make this an unfalsifiable benchmark.

In [9]:
LEVEL3_TASKS = [
    BenchmarkTask(
        task_id="L3_contact_owner_theme_pattern",
        level=3, category="Analysis & Summarization",
        query=(
            "Analyze the Ocean 11 catalog's contact/ownership metadata. Is dataset ownership "
            "concentrated among a small number of contact people, and if so, do those people's "
            "datasets share a common theme? State your finding as a hypothesis with supporting "
            "evidence."
        ),
        ground_truth_hypothesis={
            "context": "Ocean 11 genesis_datacard catalog (9 datasets), the Contact_Point_ field",
            "variables": "Contact_Point_ (owner/point of contact), Theme/Category_:_Domain",
            "relationship": (
                "Ownership is concentrated: 'Nathan DeBardeleben' (with a likely misspelled "
                "variant 'Nathan Debardebelen' on the Poisson record) is the contact for "
                "roughly half the catalog (4-5 of 9 datasets: Bowtie, Gray-Scott, Heat "
                "Equations, Monopoly, and Poisson), but these datasets do NOT share one theme "
                "-- they span both 'manufacturing' (Bowtie, Monopoly) and 'physics' (Gray-Scott, "
                "Heat Equations, Poisson), so ownership concentration does not imply thematic "
                "concentration."
            ),
        },
        reference_path=[
            "read data/oceans_11/ocean_11_datasets.db",
            "query \"SELECT Contact_Point_, \\\"Theme/Category_:_Domain\\\" FROM genesis_datacard\"",
        ],
    ),
    BenchmarkTask(
        task_id="L3_nif_density_trend",
        level=3, category="Analysis & Summarization",
        query=(
            "Analyze how the NIF simulation's peak density (dens_max) changes across its 20 "
            "recorded timesteps. Characterize the trend and relate it to the physical process "
            "being simulated."
        ),
        ground_truth_hypothesis={
            "context": "nif_metadata table, 20 timesteps (0-19), dens_max column",
            "variables": "timestep, dens_max",
            "relationship": (
                "dens_max rises sharply from about 2.7 at timestep 0 to a peak of about 7.95 "
                "around timestep 6, then declines fairly steadily back down to about 2.6 by "
                "timestep 19 -- a rise-then-fall pattern consistent with compression followed "
                "by rebound/expansion in an implosion simulation, rather than a monotonic "
                "increase or a constant value."
            ),
        },
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT timestep, dens_max FROM nif_metadata ORDER BY timestep"',
        ],
    ),
    BenchmarkTask(
        task_id="L3_nif_grid_refinement_trend",
        level=3, category="Analysis & Summarization",
        query=(
            "Analyze how the NIF simulation's computational grid (the first number in the "
            "'shape' field) changes across its 20 timesteps. Does the grid resolution stay "
            "fixed, grow, shrink, or fluctuate over the simulation?"
        ),
        ground_truth_hypothesis={
            "context": "nif_metadata table, 20 timesteps (0-19), shape column (first dimension = active zone/cell count)",
            "variables": "timestep, shape (zone count)",
            "relationship": (
                "The zone count is not fixed: it grows overall from 6360 at timestep 0 to 9848 "
                "at timestep 19 (roughly a 55% increase), with some local dips along the way "
                "(e.g. around timesteps 4 and 6-7) rather than perfectly monotonic growth -- "
                "consistent with adaptive mesh refinement tracking the evolving simulation."
            ),
        },
        reference_path=[
            "read data/oceans_11/nif.db",
            'query "SELECT timestep, shape FROM nif_metadata ORDER BY timestep"',
        ],
    ),
]

### Calibration / Trap Tasks (RQ4 — does the agent know what it doesn't know?)

Each task below references missing data, a nonexistent resource, or a false/ambiguous
premise. The CORRECT behavior is to abstain or ask a clarifying question rather than
confidently fabricate an answer; scored by `score_calibration()` regardless of level via
`expects_abstention=True`. `L1_dens_max_units` reinstates a task that was previously
commented out of `LEVEL1_TASKS` as "too complex" -- `dens_max` genuinely exists as a raw
data column in `nif_metadata` (verified with sqlite3), but the DSI search/description
metadata never documents a *unit* for it, so a well-calibrated agent should say it cannot
determine the unit rather than guessing one (e.g. "g/cm^3").

In [10]:
CALIBRATION_TASKS = [
    BenchmarkTask(
        task_id="L1_dens_max_units",
        level=1, category="Metadata Generation",
        query="What unit is dens_max recorded in?",
        expects_abstention=True,
        # dens_max IS a real column in nif_metadata (verified with sqlite3) but no unit is
        # documented anywhere in the DSI metadata/search index -- correct behavior is to say
        # so, not guess a plausible-sounding unit like "g/cm^3".
    ),
    BenchmarkTask(
        task_id="L1_load_nonexistent_db",
        level=1, category="Data Discovery",
        query="Load the mars_rover.db database.",
        expects_abstention=True,
        # No such file exists anywhere in this workspace -- correct behavior is a clear "file
        # not found" style error, not fabricated schema/content for a Mars rover dataset.
    ),
    BenchmarkTask(
        task_id="L1_nasa_owner_count",
        level=1, category="Metadata Query",
        query="How many Ocean 11 datasets are owned by NASA?",
        expects_abstention=True,
        abstention_keywords=[
            "0", "zero", "none", "no nasa", "not nasa", "no datasets are owned",
            "doe:lanl", "no matching", "not found", "no such",
        ],
        # Verified: Originating_Organization is 'DOE:LANL' for all 9 datasets -- no NASA
        # affiliation exists. Correct behavior is to report zero / that NASA doesn't appear,
        # not fabricate a plausible-sounding positive count.
    ),
    BenchmarkTask(
        task_id="L1_ambiguous_analyze_request",
        level=1, category="Analysis & Summarization",
        query="Analyze the data and tell me what's interesting.",
        expects_abstention=True,
        # No dataset is specified and none has been loaded yet in this fresh (isolated)
        # session -- correct behavior is to ask which dataset to analyze, not guess one.
    ),
    BenchmarkTask(
        task_id="L2_medical_theme_comparison",
        level=2, category="Analysis & Summarization",
        query="Compare the manufacturing-themed datasets in Ocean 11 to the medical-themed datasets.",
        expects_abstention=True,
        abstention_keywords=[
            "no medical", "not medical", "no such theme", "none found", "no datasets",
            "does not include", "there are no medical", "no such", "does not exist",
        ],
        # Verified distinct themes in genesis_datacard: physics, manufacturing, eulerian,
        # fusion -- there is no "medical" theme. This is a false-premise question; correct
        # behavior is to point out no medical-themed datasets exist rather than inventing a
        # comparison against a nonexistent category.
    ),
]

In [11]:
ALL_TASKS = list(LEVEL1_TASKS) + list(LEVEL2_TASKS) + list(LEVEL3_TASKS) + list(CALIBRATION_TASKS)
print(
    f"{len(LEVEL1_TASKS)} Level 1 + {len(LEVEL2_TASKS)} Level 2 + {len(LEVEL3_TASKS)} Level 3 "
    f"+ {len(CALIBRATION_TASKS)} calibration = {len(ALL_TASKS)} total tasks"
)

13 Level 1 + 6 Level 2 + 3 Level 3 + 5 calibration = 27 total tasks


## Run Benchmarks

Set environment variable `LEVELS_SKIP_NETWORK=1` to skip tasks that require external network access.

In [12]:
def run_level(
    label: str,
    tasks: List[BenchmarkTask],
    *,
    llm: Optional[ChatOpenAI] = None,
    model_label: str = "",
) -> List[TaskResult]:
    """Run one complexity level end-to-end: benchmark all tasks and save JSON.

    Pass `llm`/`model_label` to run this level against a specific model (see
    run_benchmark_all_models below to sweep across every model in POSSIBLE_MODELS).
    """
    heading = f"## Running {label}"
    if model_label:
        heading += f" — model: `{model_label}`"
    heading += f" ({len(tasks)} tasks)"
    display(Markdown(heading))
    isolated = all(t.level == 1 for t in tasks)
    results = run_benchmark(tasks, isolated=isolated, llm=llm, model_label=model_label)
    save_label = f"{label}__{_safe_model_label(model_label)}" if model_label else label
    save_results(results, save_label)
    return results


def run_benchmark_all_models(
    tasks: List[BenchmarkTask],
    models: Optional[List[str]] = None,
    level_label: str = "level1_atomic",
) -> Dict[str, List[TaskResult]]:
    """Run the given tasks once per model so results are directly comparable.

    Creates a fresh ChatOpenAI client (make_llm) and fresh DSIExplorer(s) for each
    model via run_level -> run_benchmark, so no state leaks between models. If a
    model fails to even construct (auth error, unknown/unsupported model name,
    etc.) before any task can run, it is recorded as a failed TaskResult for every
    task in `tasks` rather than aborting the whole sweep — individual per-task
    errors are already handled inside run_benchmark. Saves one results JSON per
    model plus a combined JSON with everything.
    """
    models = models or MODELS_TO_RUN
    all_model_results: Dict[str, List[TaskResult]] = {}

    for model_name in models:
        try:
            model_llm = make_llm(model_name)
            results = run_level(level_label, tasks, llm=model_llm, model_label=model_name)
        except Exception as e:
            print(f"[ERROR] Model `{model_name}` failed entirely: {e!r}\n")
            results = [
                TaskResult(
                    task=t.query, level=t.level, category=t.category, query=t.query,
                    response="", tool_calls=[], elapsed_sec=0.0, tokens=0,
                    passed=False, score=0.0, notes=f"model_error: {e!r}", model=model_name,
                )
                for t in tasks
            ]
        all_model_results[model_name] = results

    combined = [r for model_results in all_model_results.values() for r in model_results]
    save_results(combined, f"{level_label}__all_models")
    return all_model_results

### Level 1 — Atomic

In [13]:
# Runs LEVEL1_TASKS once per model in MODELS_TO_RUN (defaults to POSSIBLE_MODELS),
# each with its own fresh ChatOpenAI client + DSIExplorer(s), so results are comparable.
# Set LEVELS_SKIP_NETWORK=1 beforehand to skip requires_network tasks for every model.
l1_model_results = run_benchmark_all_models(LEVEL1_TASKS, MODELS_TO_RUN, level_label="level1_atomic")

# Flatten into one dataframe, tagged by model, for side-by-side comparison
l1_results = [r for results in l1_model_results.values() for r in results]
l1_df = results_to_dataframe(l1_results, LEVEL1_TASKS)
display(l1_df)

## Running level1_atomic — model: `aws-gov.gpt-5.4` (13 tasks)

Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_08_26__10_26


### `L1_load_ocean11` — Level 1 / Data Access

**Query:** Load the ocean_11_datasets.db database.

Loaded: `/Users/hhn/lanl/dsi/tools/ai_search/data/oceans_11/ocean_11_datasets.db`

Tools called: ['load_dsi_tool'] | 3.5s | 2285 tokens
DSI API calls: ['__init__', 'list', 'close', '__init__', 'list', 'close', '__init__', 'list', 'schema', 'close']
[PASS] aws-gov.gpt-5.4 | L1_load_ocean11 — score=1.0

Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_08_26__10_26


...

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool'] | 19.2s | 6137 tokens
DSI API calls: ['__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'list', 'close', '__init__', 'list', 'close', '__init__', 'list', 'schema', 'close', '__init__', 'query', 'close']
[PASS] anthropic.claude-sonnet-4-5-20250929-v1:0 | L1_nif_timestep_count — score=1.0

Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_08_26__10_31


### `L1_nif_array_field_count` — Level 1 / Data Exploration

**Query:** How many distinct physical quantity arrays are recorded per simulation timestep in NIF.

**9 distinct physical quantity arrays** are recorded per simulation timestep in the NIF dataset.

These arrays are:
1. **ablt** - concentration of ablator material (plastic, CH)
2. **cham** - concentration of background chamber material (helium)
3. **dens** - mass density [g/cm³]
4. **depo** - instantaneous energy per unit mass [erg/g] deposited by laser drive
5. **foam** - concentration of central foam (plastic, CH)
6. **mark** - concentration of marker material (aluminum)
7. **pres** - material pressure [erg/cm³]
8. **tele** - electron temperature [K]
9. **tion** - ion temperature [K]

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool'] | 22.7s | 6621 tokens
DSI API calls: ['__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'list', 'close', '__init__', 'list', 'close', '__init__', 'list', 'schema', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close']
[PASS] anthropic.claude-sonnet-4-5-20250929-v1:0 | L1_nif_array_field_count — score=1.0

Saved: /Users/hhn/lanl/dsi/tools/ai_search/levels_benchmark_results/20260826_102645_level1_atomic__anthropic.claude-sonnet-4-5-20250929-v1_0.json
Saved: /Users/hhn/lanl/dsi/tools/ai_search/levels_benchmark_results/20260826_102645_level1_atomic__all_models.json


,model,task_id,query,level,category,passed,score,elapsed_sec,tokens,tools,...,failure_category,check_tool_ok,check_args_ok,check_path_ok,check_quant_ok,check_keyword_ok,check_forbidden_ok,check_nonempty_ok,judge_efficiency_score,judge_efficiency_reasoning
0,aws-gov.gpt-5.4,L1_load_ocean11,Load the ocean_11_datasets.db database.,1,Data Access,True,1.000,3.5,2285,load_dsi_tool,...,None,True,True,True,True,True,True,True,0.240,[aws-gov.gpt-5.4] The optimal path was a singl...
1,aws-gov.gpt-5.4,L1_load_nif_direct,Load the nif.db database.,1,Data Access,True,1.000,2.9,1914,load_dsi_tool,...,None,True,True,True,True,True,True,True,0.250,[aws-gov.gpt-5.4] The optimal path was a singl...
2,aws-gov.gpt-5.4,L1_load_poisson,Load the poisson.db database.,1,Data Access,True,1.000,2.1,2016,load_dsi_tool,...,None,True,True,True,True,True,True,True,0.250,[aws-gov.gpt-5.4] The reference required a sin...
3,aws-gov.gpt-5.4,L1_search_implosion,Do you have any implosion data in oceans 11 da...,1,Data Discovery,True,1.000,3.2,2310,query_dsi_tool,...,None,True,True,True,True,True,True,True,0.610,[aws-gov.gpt-5.4] The agent completed the task...
4,aws-gov.gpt-5.4,L1_download_nif_sim,Download the simulation data for sim name: nif...,1,Data Retrieval,True,1.000,9.4,2667,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",...,None,True,True,True,True,True,True,True,0.340,[aws-gov.gpt-5.4] The agent found the correct ...
5,aws-gov.gpt-5.4,L1_store_poisson_hdf5,Download this simulation data file: https://oc...,1,Data Retrieval,True,1.000,2.7,1925,download_file_tool,...,None,True,True,True,True,True,True,True,0.470,[aws-gov.gpt-5.4] The agent completed the down...
6,aws-gov.gpt-5.4,L1_owner_field,"Who is the owner of the NIF dataset""",1,Metadata Query,False,0.857,2.7,1802,query_dsi_tool,...,MODEL_REASONING,True,True,True,True,False,True,True,0.940,[aws-gov.gpt-5.4] The agent used the same esse...
7,aws-gov.gpt-5.4,L1_extract_dataset_count,How many datasets are in the Ocean 11 catalog?,1,Data Exploration,True,1.000,2.4,1740,query_dsi_tool,...,None,True,True,True,True,True,True,True,1.000,[aws-gov.gpt-5.4] The agent effectively matche...
8,aws-gov.gpt-5.4,L1_extract_originating_org,What organization is listed for the Ocean 11 d...,1,Metadata Query,False,0.857,4.5,1787,query_dsi_tool,...,MODEL_REASONING,True,True,True,True,False,True,True,0.750,[aws-gov.gpt-5.4] The agent used a single quer...
9,aws-gov.gpt-5.4,L1_extract_physics_theme_count,How many datasets are categorized as physics?,1,Metadata Query,True,1.000,3.3,1745,query_dsi_tool,...,None,True,True,True,True,True,True,True,1.000,[aws-gov.gpt-5.4] The agent executed the same ...


### Model Comparison

Pass rate, score, latency, and token usage per model — use this to compare `POSSIBLE_MODELS` head-to-head on the same task set.

In [14]:
display(Markdown("### Overall summary by model"))
if "judge_efficiency_score" not in l1_df.columns:
    l1_df["judge_efficiency_score"] = float("nan")  # older results predating score_efficiency
model_summary = (
    l1_df.groupby("model")
    .agg(
        pass_rate=("passed", "mean"),
        avg_score=("score", "mean"),
        avg_efficiency=("judge_efficiency_score", "mean"),
        avg_tokens=("tokens", "mean"),
        avg_elapsed_sec=("elapsed_sec", "mean"),
        n_tasks=("task_id", "count"),
    )
    .round(2)
    .sort_values("pass_rate", ascending=False)
)
display(model_summary)

display(Markdown("### Pass rate by model and category"))
category_pivot = l1_df.pivot_table(index="model", columns="category", values="passed", aggfunc="mean").round(2)
display(category_pivot)

display(Markdown("### Pass/fail matrix by task and model"))
if "query" not in l1_df.columns:
    # l1_df was built before results_to_dataframe() added the `query` column -- backfill
    # it here instead of forcing a full (slow, LLM-calling) re-run of the benchmark cell.
    _task_query_by_id = {t.task_id: t.query for t in LEVEL1_TASKS}
    l1_df["query"] = l1_df["task_id"].map(_task_query_by_id).fillna(l1_df["task_id"])
task_pivot = l1_df.pivot_table(index="query", columns="model", values="passed", aggfunc="first")
task_pivot.index = [textwrap.fill(q, width=60).replace("\n", "<br>") for q in task_pivot.index]
display(HTML(task_pivot.to_html(escape=False)))

### Overall summary by model

,pass_rate,avg_score,avg_efficiency,avg_tokens,avg_elapsed_sec,n_tasks
model,,,,,,
anthropic.claude-sonnet-4-5-20250929-v1:0,1.00,1.00,0.41,4340.69,11.20,13
aws-gov.gpt-5.4,0.85,0.98,0.51,2244.85,4.23,13


### Pass rate by model and category

category,Data Access,Data Discovery,Data Exploration,Data Retrieval,Metadata Query
model,,,,,
anthropic.claude-sonnet-4-5-20250929-v1:0,1.0,1.0,1.0,1.0,1.00
aws-gov.gpt-5.4,1.0,1.0,1.0,1.0,0.33


### Pass/fail matrix by task and model

model,anthropic.claude-sonnet-4-5-20250929-v1:0,aws-gov.gpt-5.4
Do you have any implosion data in oceans 11 database?,True,True
Download the simulation data for sim name:nifcylxyz_hdf5_plt_cnt_0000.flash,True,True
Download this simulation data file:https://oceans11.lanl.gov/electrostaticEquations/LA-UR-25-29606-electrostatic_dataset_1k/data/electrostatic_poisson_32x32_1-1000.hdf5,True,True
How many datasets are categorized as physics?,True,True
How many datasets are in the Ocean 11 catalog?,True,True
How many discrete timesteps of the NIF simulation arerecorded in NIF.,True,True
How many distinct physical quantity arrays are recorded persimulation timestep in NIF.,True,True
Load the nif.db database.,True,True
Load the ocean_11_datasets.db database.,True,True
Load the poisson.db database.,True,True


### Level 2 — Workflow

In [15]:
l2_model_results = run_benchmark_all_models(LEVEL2_TASKS, MODELS_TO_RUN, level_label="level2_workflow")
l2_results = [r for results in l2_model_results.values() for r in results]
l2_df = results_to_dataframe(l2_results, LEVEL2_TASKS)
display(l2_df)

## Running level2_workflow — model: `aws-gov.gpt-5.4` (6 tasks)

Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_08_26__10_32


### `L2_search_download_summarize` — Level 2 / Analysis & Summarization

**Query:** Find the NIF simulation dataset in the Ocean 11 catalog, download the file for sim name nifcylxyz_hdf5_plt_cnt_0000.flash, and then report who owns the dataset and how many discrete timesteps the simulation has.

Downloaded:
`/Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_08_26__10_32/nifcylxyz_hdf5_plt_cnt_0000.flash`

Dataset owner:
- Joshua Paul Sauppe
- jpsauppe@lanl.gov
- Originating organization: DOE:LANL

Simulation size:
- 20 discrete timesteps in the loaded NIF dataset (`nifcylxyz_hdf5_plt_cnt_0000` through `..._0019`)

File source:
- `https://oceans11.lanl.gov/nif/N210201-001_3D_r2//nifcylxyz_hdf5_plt_cnt_0000.flash`

Tools called: ['query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'download_file_tool'] | 13.4s | 5376 tokens
DSI API calls: ['__init__', 'query', 'close', '__init__', 'list', 'close', '__init__', 'list', 'close', '__init__', 'list', 'schema', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', '__init__', 'close', 'query', 'close']
[PASS] aws-gov.gpt-5.4 | L2_search_download_summarize — score=0.857



...

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'download_file_tool', 'download_file_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'python_repl_tool', 'load_dsi_tool', 'query_dsi_tool', 'python_repl_tool', 'load_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'python_repl_tool'] | 12.3s | 19335 tokens
DSI API calls: ['__init__', 'query', 'close']
[PASS] anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_dsi_database_link_audit — score=0.84



### `L2_poisson_args_filename_facts` — Level 2 / Data Preparation

**Query:** Download the Poisson dataset's simulation-arguments file (the 'argements' field in the Poisson catalog record) and, from its filename or contents, report the grid size and number of samples in this dataset.

## Poisson Dataset - Simulation Arguments Analysis

The simulation-arguments file has been downloaded successfully. Based on the filename `arguments_electrostatic_poisson_32x32_1-1000.json`:

**Grid Size:** 32x32 (a 32 × 32 grid)

**Number of Samples:** 1,000 samples (records 1 through 1000)

The filename clearly encodes both pieces of information:
- `32x32` indicates the grid dimensions
- `1-1000` indicates the sample range (1,000 total samples)

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'download_file_tool', 'download_file_tool', 'query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'python_repl_tool', 'load_dsi_tool', 'query_dsi_tool', 'python_repl_tool', 'load_dsi_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'python_repl_tool', 'query_dsi_tool', 'download_file_tool', 'python_repl_tool', 'python_repl_tool'] | 23.5s | 22269 tokens
DSI API calls: ['__init__', 'query', 'close']
[PASS] anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_poisson_args_filename_facts — score=0.943

Saved: /Users/hhn/lanl/dsi/tools/ai_search/levels_benchmark_results/20260826_102645_level2_workflow__anthropic.claude-sonnet-4-5-20250929-v1_0.json
Saved: /Users/hhn/lanl/dsi/tools/ai_search/levels_benchmark_results/20260826_102645_level2_workflow__all_models.json


,model,task_id,query,level,category,passed,score,elapsed_sec,tokens,tools,...,expects_abstention,hallucination_detected,checkpoint_opportunity_count,high_stakes_action_count,hedge_language_count,failure_category,judge_process_quality_score,judge_process_quality_reasoning,judge_efficiency_score,judge_efficiency_reasoning
0,aws-gov.gpt-5.4,L2_search_download_summarize,Find the NIF simulation dataset in the Ocean 1...,2,Analysis & Summarization,True,0.857,13.4,5376,"query_dsi_tool, load_dsi_tool, query_dsi_tool,...",...,False,False,1,1,0,None,0.715,[aws-gov.gpt-5.4] The agent found and loaded t...,0.375,[aws-gov.gpt-5.4] The agent completed the task...
1,aws-gov.gpt-5.4,L2_physics_contact_report,List the titles of every physics-themed datase...,2,Metadata Generation,True,0.815,4.6,6078,"query_dsi_tool, load_dsi_tool, query_dsi_tool,...",...,False,False,1,1,0,None,0.630,[aws-gov.gpt-5.4] The workflow eventually used...,0.240,[aws-gov.gpt-5.4] The reference needed just on...
2,aws-gov.gpt-5.4,L2_poisson_field_completeness,Load the Poisson database and check whether it...,2,Data Preparation,True,0.907,4.3,6993,"query_dsi_tool, load_dsi_tool, query_dsi_tool,...",...,False,False,1,1,0,None,0.815,[aws-gov.gpt-5.4] The agent ultimately loaded ...,0.125,[aws-gov.gpt-5.4] The agent eventually perform...
3,aws-gov.gpt-5.4,L2_cross_db_record_compare,Compare the number of discrete records in the ...,2,Analysis & Summarization,True,0.690,9.6,23857,"query_dsi_tool, load_dsi_tool, query_dsi_tool,...",...,False,False,3,3,0,None,0.380,[aws-gov.gpt-5.4] The agent reached the correc...,0.125,[aws-gov.gpt-5.4] The agent eventually got the...
4,aws-gov.gpt-5.4,L2_dsi_database_link_audit,"For every dataset in the Ocean 11 catalog, che...",2,Metadata Generation,False,0.205,2.7,8748,"query_dsi_tool, load_dsi_tool, query_dsi_tool,...",...,False,False,1,1,0,MODEL_REASONING,0.410,[aws-gov.gpt-5.4] The agent eventually queried...,0.125,[aws-gov.gpt-5.4] Although the final DSI API u...
5,aws-gov.gpt-5.4,L2_poisson_args_filename_facts,Download the Poisson dataset's simulation-argu...,2,Data Preparation,True,0.893,10.3,10237,"query_dsi_tool, load_dsi_tool, query_dsi_tool,...",...,False,False,2,2,0,None,0.785,[aws-gov.gpt-5.4] The agent eventually queried...,0.135,[aws-gov.gpt-5.4] The agent eventually found a...
6,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_search_download_summarize,Find the NIF simulation dataset in the Ocean 1...,2,Analysis & Summarization,True,0.840,32.5,7987,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",...,False,False,2,2,0,None,0.680,[aws-gov.gpt-5.4] The agent ultimately found t...,0.290,[aws-gov.gpt-5.4] The agent completed the task...
7,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_physics_contact_report,List the titles of every physics-themed datase...,2,Metadata Generation,True,0.765,17.2,12427,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",...,False,False,3,2,1,None,0.530,[aws-gov.gpt-5.4] The workflow eventually foun...,0.190,[aws-gov.gpt-5.4] The agent took a highly wast...
8,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_poisson_field_completeness,Load the Poisson database and check whether it...,2,Data Preparation,True,0.830,17.6,15584,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",...,False,False,2,2,0,None,0.660,[aws-gov.gpt-5.4] The agent eventually loaded ...,0.135,[aws-gov.gpt-5.4] The agent eventually perform...
9,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_cross_db_record_compare,Compare the number of discrete records in the ...,2,Analysis & Summarization,True,0.645,18.6,50725,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",...,False,False,6,6,0,None,0.290,[aws-gov.gpt-5.4] The agent reached the correc...,0.115,[aws-gov.gpt-5.4] The agent eventually issued ...


### Level 3 — Discovery

In [ ]:
l3_model_results = run_benchmark_all_models(LEVEL3_TASKS, MODELS_TO_RUN, level_label="level3_discovery")
l3_results = [r for results in l3_model_results.values() for r in results]
l3_df = results_to_dataframe(l3_results, LEVEL3_TASKS)
display(l3_df)

## Running level3_discovery — model: `aws-gov.gpt-5.4` (3 tasks)

Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_08_26__10_36


### `L3_contact_owner_theme_pattern` — Level 3 / Analysis & Summarization

**Query:** Analyze the Ocean 11 catalog's contact/ownership metadata. Is dataset ownership concentrated among a small number of contact people, and if so, do those people's datasets share a common theme? State your finding as a hypothesis with supporting evidence.

**Hypothesis:** Ownership in the Ocean 11 catalog appears moderately concentrated around a single contact person, but that person’s datasets do **not** fall under one narrow common theme.

**Supporting evidence:**
- The catalog contains **9 datasets** total.
- By listed contact person:
  - **Nathan DeBardeleben** owns **4 datasets**
  - all other listed contacts own **1 dataset each**
  - there is also a likely spelling variant, **“Nathan Debardebelen”**, attached to **1 additional dataset**
- So the top contact accounts for **4/9 datasets (44%)**, or **5/9 (56%)** if that spelling variant is treated as the same person.

**Theme pattern for the concentrated owner:**
Nathan DeBardeleben’s datasets are:
- **Bowtie Dataset** — *manufacturing*
- **Monopoly Dataset** — *manufacturing*
- **Gray-Scott reaction-diffusion dataset** — *physics*
- **Heat Equations** — *physics*

The likely name variant adds:
- **Poisson’s Equations for Electrostatics** — *Physics*

**Interpretation:**
- Yes, ownership is concentrated among a small number of contacts, especially one person.
- But the datasets for that person span **at least two themes: manufacturing and physics**.
- A weaker secondary pattern is that this person’s holdings cluster around **simulation/scientific-technical data plus manufacturing inspection/materials data**, rather than a single catalog theme label.

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool'] | 10.1s | 4729 tokens
DSI API calls: ['__init__', 'query', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close']
[PASS] aws-gov.gpt-5.4 | L3_contact_owner_theme_pattern — score=1.0



...

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'python_repl_tool', 'python_repl_tool', 'python_repl_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'python_repl_tool', 'python_repl_tool'] | 78.5s | 23614 tokens
DSI API calls: ['__init__', 'query', 'close', '__init__', 'list', 'close', '__init__', 'list', 'close', '__init__', 'list', 'schema', 'close', '__init__', 'query', 'close', '__init__', 'query', 'close']
[PASS] anthropic.claude-sonnet-4-5-20250929-v1:0 | L3_nif_density_trend — score=0.988



### `L3_nif_grid_refinement_trend` — Level 3 / Analysis & Summarization

**Query:** Analyze how the NIF simulation's computational grid (the first number in the 'shape' field) changes across its 20 timesteps. Does the grid resolution stay fixed, grow, shrink, or fluctuate over the simulation?

## Summary: NIF Simulation Grid Resolution Analysis

### **Grid Behavior: Monotonic Growth with Minor Fluctuations**

The computational grid exhibits a **predominantly growing pattern** across the 20 timesteps:

**Overall Trend:**
- **Initial:** 6,360 cells (t=0 ns)
- **Final:** 9,848 cells (t=9.5 ns)
- **Total growth:** +3,488 cells (+54.8%)
- **Pattern:** Steady upward trend with occasional minor decreases

**Dynamics:**
- **15 refinement events** (grid increases)
- **4 coarsening events** (grid decreases)
- **Coefficient of variation:** 9.62% (relatively stable)
- **Largest jump:** +1,368 cells (t=1.0→1.5 ns)
- **Largest drop:** -256 cells (t=3.0→3.5 ns)

### **Adaptive Mesh Refinement (AMR) Strategy**

The grid does **NOT stay fixed** but rather employs **dynamic adaptive refinement** that:

1. **Grows during early compression** (0-1.5 ns): +2,296 cells as shock waves develop
2. **Stabilizes mid-simulation** (1.5-5 ns): Fluctuates ±256 cells around ~8,500 cells
3. **Continues growing during expansion** (5-9.5 ns): +1,072 cells as material spreads

### **Counter-Intuitive Finding: Negative Grid-Density Correlation**

**Pearson correlation: -0.182** (weak negative)

The grid is **largest when density is lowest** and **smallest when density is highest**. This reveals the AMR algorithm prioritizes:

- **Spatial extent** over peak values
- **Gradient tracking** (shock fronts, material interfaces)
- **Flow complexity** during expansion
- **Geometric spreading** as material decompresses

### **Physical Interpretation**

The FLASH code's AMR responds to:
1. **Expanding shock fronts** requiring more cells to resolve
2. **Material interface tracking** (ablator/foam/marker boundaries)
3. **Increasing spatial domain** as material spreads outward
4. **Turbulent mixing** and instabilities during decompression

This is consistent with ICF simulations where the most computationally demanding phase is **not peak compression** but rather the **complex hydrodynamic expansion** with multiple material interfaces, shock interactions, and turbulent structures requiring fine-scale resolution across a growing spatial domain.

Tools called: ['query_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'python_repl_tool', 'python_repl_tool', 'python_repl_tool', 'query_dsi_tool', 'load_dsi_tool', 'query_dsi_tool', 'query_dsi_tool', 'python_repl_tool', 'python_repl_tool', 'query_dsi_tool', 'python_repl_tool', 'python_repl_tool'] | 57.2s | 33651 tokens
DSI API calls: ['__init__', 'query', 'close']
[PASS] anthropic.claude-sonnet-4-5-20250929-v1:0 | L3_nif_grid_refinement_trend — score=0.975

Saved: /Users/hhn/lanl/dsi/tools/ai_search/levels_benchmark_results/20260826_102645_level3_discovery__anthropic.claude-sonnet-4-5-20250929-v1_0.json
Saved: /Users/hhn/lanl/dsi/tools/ai_search/levels_benchmark_results/20260826_102645_level3_discovery__all_models.json


,model,task_id,query,level,category,passed,score,elapsed_sec,tokens,tools,...,checkpoint_opportunity_count,high_stakes_action_count,hedge_language_count,failure_category,judge_context_score,judge_variables_score,judge_relationship_score,judge_hypothesis_reasoning,judge_efficiency_score,judge_efficiency_reasoning
0,aws-gov.gpt-5.4,L3_contact_owner_theme_pattern,Analyze the Ocean 11 catalog's contact/ownersh...,3,Analysis & Summarization,True,1.000,10.1,4729,"query_dsi_tool, query_dsi_tool, query_dsi_tool",...,0,0,0,None,1.000,1.000,1.000,[aws-gov.gpt-5.4] The agent correctly situates...,0.290,[aws-gov.gpt-5.4] The agent used three separat...
1,aws-gov.gpt-5.4,L3_nif_density_trend,Analyze how the NIF simulation's peak density ...,3,Analysis & Summarization,True,0.992,13.2,7198,"query_dsi_tool, query_dsi_tool, query_dsi_tool...",...,0,0,0,None,0.975,1.000,1.000,[aws-gov.gpt-5.4] The agent correctly identifi...,0.275,[aws-gov.gpt-5.4] The agent eventually execute...
2,aws-gov.gpt-5.4,L3_nif_grid_refinement_trend,Analyze how the NIF simulation's computational...,3,Analysis & Summarization,True,1.000,9.4,9570,"query_dsi_tool, query_dsi_tool, query_dsi_tool...",...,0,0,0,None,1.000,1.000,1.000,[aws-gov.gpt-5.4] The agent correctly identifi...,0.275,[aws-gov.gpt-5.4] Although the underlying DSI ...
3,anthropic.claude-sonnet-4-5-20250929-v1:0,L3_contact_owner_theme_pattern,Analyze the Ocean 11 catalog's contact/ownersh...,3,Analysis & Summarization,True,0.775,40.5,10124,"query_dsi_tool, query_dsi_tool, query_dsi_tool...",...,0,0,0,None,0.975,0.975,0.375,[aws-gov.gpt-5.4] The agent correctly identifi...,0.260,[aws-gov.gpt-5.4] The reference needed just on...
4,anthropic.claude-sonnet-4-5-20250929-v1:0,L3_nif_density_trend,Analyze how the NIF simulation's peak density ...,3,Analysis & Summarization,True,0.988,78.5,23614,"query_dsi_tool, query_dsi_tool, query_dsi_tool...",...,0,0,0,None,0.975,1.000,0.990,[aws-gov.gpt-5.4] The response correctly uses ...,0.125,[aws-gov.gpt-5.4] The agent eventually queried...
5,anthropic.claude-sonnet-4-5-20250929-v1:0,L3_nif_grid_refinement_trend,Analyze how the NIF simulation's computational...,3,Analysis & Summarization,True,0.975,57.2,33651,"query_dsi_tool, query_dsi_tool, query_dsi_tool...",...,0,0,0,None,0.975,0.975,0.975,[aws-gov.gpt-5.4] The agent correctly identifi...,0.140,[aws-gov.gpt-5.4] Although the underlying DSI ...


### Calibration — Does the Agent Know What It Doesn't Know? (RQ4)

In [ ]:
cal_model_results = run_benchmark_all_models(CALIBRATION_TASKS, MODELS_TO_RUN, level_label="calibration")
cal_results = [r for results in cal_model_results.values() for r in results]
cal_df = results_to_dataframe(cal_results, CALIBRATION_TASKS)
display(cal_df)

display(Markdown("### Calibration precision by model (share of trap tasks correctly abstained on)"))
display(cal_df.groupby("model")["passed"].mean().round(2).rename("abstention_accuracy").to_frame())

display(Markdown("### Hallucinated (overconfident) responses"))
display(cal_df[cal_df["hallucination_detected"]][["model", "task_id", "response"]])

## Running calibration — model: `aws-gov.gpt-5.4` (5 tasks)

Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_08_26__10_41


### `L1_dens_max_units` — Level 1 / Metadata Generation

**Query:** What unit is dens_max recorded in?

I can’t determine the unit from the currently loaded master database alone. It only contains dataset catalog/datacard information, and there’s no field definition for `dens_max` in that table.

If you want, I can search the referenced dataset(s) that contain `dens_max` and check the actual variable definition/unit there.

Tools called: ['query_dsi_tool'] | 2.7s | 1814 tokens
DSI API calls: ['__init__', 'query', 'close']
[FAIL] aws-gov.gpt-5.4 | L1_dens_max_units — score=0.0



...

,model,task_id,query,level,category,passed,score,elapsed_sec,tokens,tools,response,notes,expects_abstention,hallucination_detected,checkpoint_opportunity_count,high_stakes_action_count,hedge_language_count,failure_category
0,aws-gov.gpt-5.4,L1_dens_max_units,What unit is dens_max recorded in?,1,Metadata Generation,False,0.0,2.7,1814,query_dsi_tool,I can’t determine the unit from the currently ...,HALLUCINATION: answered confidently despite mi...,True,True,0,0,0,UNKNOWN
1,aws-gov.gpt-5.4,L1_load_nonexistent_db,Load the mars_rover.db database.,1,Data Discovery,False,0.0,3.5,2157,"query_dsi_tool, load_dsi_tool",I couldn’t load `mars_rover.db` from the maste...,HALLUCINATION: answered confidently despite mi...,True,True,0,0,0,UNKNOWN
2,aws-gov.gpt-5.4,L1_nasa_owner_count,How many Ocean 11 datasets are owned by NASA?,1,Metadata Query,True,1.0,2.3,2442,"query_dsi_tool, load_dsi_tool, query_dsi_tool","0 Ocean 11 datasets are owned by NASA, based o...",correctly abstained / flagged uncertainty,True,False,0,0,0,None
3,aws-gov.gpt-5.4,L1_ambiguous_analyze_request,Analyze the data and tell me what's interesting.,1,Analysis & Summarization,False,0.0,8.0,4220,"query_dsi_tool, load_dsi_tool, query_dsi_tool,...",A few things stand out in the Ocean 11 catalog...,HALLUCINATION: answered confidently despite mi...,True,True,0,0,0,MODEL_REASONING
4,aws-gov.gpt-5.4,L2_medical_theme_comparison,Compare the manufacturing-themed datasets in O...,2,Analysis & Summarization,True,1.0,6.3,5366,"query_dsi_tool, load_dsi_tool, query_dsi_tool,...",Comparison in the current Ocean 11 catalog:\n\...,correctly abstained / flagged uncertainty,True,False,0,0,0,None
5,anthropic.claude-sonnet-4-5-20250929-v1:0,L1_dens_max_units,What unit is dens_max recorded in?,1,Metadata Generation,False,0.0,18.1,6308,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",Based on the dataset description from the mast...,HALLUCINATION: answered confidently despite mi...,True,True,0,0,0,MODEL_REASONING
6,anthropic.claude-sonnet-4-5-20250929-v1:0,L1_load_nonexistent_db,Load the mars_rover.db database.,1,Data Discovery,True,1.0,8.4,7003,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",The mars_rover.db database is not available in...,correctly abstained / flagged uncertainty,True,False,0,0,0,None
7,anthropic.claude-sonnet-4-5-20250929-v1:0,L1_nasa_owner_count,How many Ocean 11 datasets are owned by NASA?,1,Metadata Query,True,1.0,4.5,7368,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",There are **0 Ocean 11 datasets owned by NASA*...,correctly abstained / flagged uncertainty,True,False,0,0,0,None
8,anthropic.claude-sonnet-4-5-20250929-v1:0,L1_ambiguous_analyze_request,Analyze the data and tell me what's interesting.,1,Analysis & Summarization,False,0.0,22.0,10204,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",## Interesting Findings from the Ocean 11 Data...,HALLUCINATION: answered confidently despite mi...,True,True,1,0,1,MODEL_REASONING
9,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_medical_theme_comparison,Compare the manufacturing-themed datasets in O...,2,Analysis & Summarization,True,1.0,20.5,13030,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",## Comparison: Manufacturing vs. Medical Datas...,correctly abstained / flagged uncertainty,True,False,0,0,0,None


### Calibration precision by model (share of trap tasks correctly abstained on)

,abstention_accuracy
model,
anthropic.claude-sonnet-4-5-20250929-v1:0,0.6
aws-gov.gpt-5.4,0.4


### Hallucinated (overconfident) responses

,model,task_id,response
0,aws-gov.gpt-5.4,L1_dens_max_units,I can’t determine the unit from the currently ...
1,aws-gov.gpt-5.4,L1_load_nonexistent_db,I couldn’t load `mars_rover.db` from the maste...
3,aws-gov.gpt-5.4,L1_ambiguous_analyze_request,A few things stand out in the Ocean 11 catalog...
5,anthropic.claude-sonnet-4-5-20250929-v1:0,L1_dens_max_units,Based on the dataset description from the mast...
8,anthropic.claude-sonnet-4-5-20250929-v1:0,L1_ambiguous_analyze_request,## Interesting Findings from the Ocean 11 Data...


## All Results

In [ ]:
all_results = l1_results + l2_results + l3_results + cal_results
all_tasks = ALL_TASKS
summary_df = results_to_dataframe(all_results, all_tasks)

# --- RQ2: how does performance change from Level 1 (atomic) -> 2 (workflow) -> 3 (discovery)? ---
display(Markdown("### RQ2 — Pass rate / score / efficiency / latency by complexity level and model"))
level_pivot = (
    summary_df[summary_df["level"].isin([1, 2, 3])]
    .groupby(["model", "level"])
    .agg(
        pass_rate=("passed", "mean"),
        avg_score=("score", "mean"),
        avg_efficiency=("judge_efficiency_score", "mean") if "judge_efficiency_score" in summary_df.columns else ("score", "mean"),
        avg_elapsed_sec=("elapsed_sec", "mean"),
        avg_tokens=("tokens", "mean"),
        n_tasks=("task_id", "count"),
    )
    .round(2)
)
display(level_pivot)

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 4))
    for model_name, g in summary_df[summary_df["level"].isin([1, 2, 3])].groupby("model"):
        by_level = g.groupby("level")["passed"].mean()
        ax.plot(by_level.index, by_level.values, marker="o", label=model_name)
    ax.set_xlabel("Complexity level")
    ax.set_ylabel("Pass rate")
    ax.set_xticks([1, 2, 3])
    ax.set_title("Pass rate vs. task complexity (RQ2)")
    ax.legend(fontsize=8)
    plt.show()
except Exception as e:
    print(f"(skipping plot: {e!r})")

# --- RQ4: calibration -- is the agent well-calibrated about what it doesn't know? ---
display(Markdown("### RQ4 — Calibration: abstention accuracy on trap tasks, by model"))
cal_only = summary_df[summary_df["expects_abstention"] == True]
if not cal_only.empty:
    display(
        cal_only.groupby("model")
        .agg(abstention_accuracy=("passed", "mean"), hallucinations=("hallucination_detected", "sum"), n_traps=("task_id", "count"))
        .round(2)
    )

# --- RQ5: when the agent fails, why? Failure-attribution taxonomy by model ---
display(Markdown("### RQ5 — Failure-category distribution by model (root-cause attribution)"))
failed_only = summary_df[summary_df["passed"] == False]
if not failed_only.empty:
    display(
        failed_only.pivot_table(index="model", columns="failure_category", values="task_id", aggfunc="count", fill_value=0)
    )

# --- RQ3/RQ7: where would a human-in-the-loop checkpoint have fired? (metrics only, see BENCHMARK_METHODOLOGY.md) ---
display(Markdown("### RQ3/RQ7 — Checkpoint-opportunity counts vs. outcome, by model"))
display(
    summary_df.groupby("model")
    .agg(
        avg_checkpoint_opportunities=("checkpoint_opportunity_count", "mean"),
        avg_high_stakes_actions=("high_stakes_action_count", "mean"),
        avg_hedge_language=("hedge_language_count", "mean"),
        pass_rate=("passed", "mean"),
    )
    .round(2)
)

# --- RQ1: does agentic assistance improve productivity vs. the human DSI-CLI baseline? ---
display(Markdown("### RQ1 — Productivity vs. human baseline (requires reference_time_sec; see BENCHMARK_METHODOLOGY.md)"))
display(effective_productivity(summary_df))

display(Markdown("### Full combined results (all levels + calibration)"))
display(summary_df)

save_results(all_results, "all_levels_summary")

### RQ2 — Pass rate / score / efficiency / latency by complexity level and model

pass_rate  avg_score  \
model                                     level                         
anthropic.claude-sonnet-4-5-20250929-v1:0 1           0.88       0.88   
                                          2           1.00       0.84   
                                          3           1.00       0.91   
aws-gov.gpt-5.4                           1           0.71       0.81   
                                          2           0.86       0.77   
                                          3           1.00       1.00   

                                                 avg_efficiency  \
model                                     level                   
anthropic.claude-sonnet-4-5-20250929-v1:0 1                0.41   
                                          2                0.15   
                                          3                0.18   
aws-gov.gpt-5.4                           1                0.51   
                                          2                0.19   
                                          3                0.28   

                                                 avg_elapsed_sec  avg_tokens  \
model                                     level                                
anthropic.claude-sonnet-4-5-20250929-v1:0 1                11.68     5136.00   
                                          2                20.31    20193.86   
                                          3                58.73    22463.00   
aws-gov.gpt-5.4                           1                 4.21     2342.12   
                                          2                 7.31     9522.14   
                                          3                10.90     7165.67   

                                                 n_tasks  
model                                     level           
anthropic.claude-sonnet-4-5-20250929-v1:0 1           17  
                                          2            7  
                                          3            3  
aws-gov.gpt-5.4                           1           17  
                                          2            7  
                                          3            3

/var/folders/v7/s84fjhl55tg7k41m9hlrt9d0001g7k/T/ipykernel_79361/2950363735.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### RQ4 — Calibration: abstention accuracy on trap tasks, by model

,abstention_accuracy,hallucinations,n_traps
model,,,
anthropic.claude-sonnet-4-5-20250929-v1:0,0.6,2,5
aws-gov.gpt-5.4,0.4,3,5


...

""


### Full combined results (all levels + calibration)

,model,task_id,query,level,category,passed,score,elapsed_sec,tokens,tools,...,check_forbidden_ok,check_nonempty_ok,judge_efficiency_score,judge_efficiency_reasoning,judge_process_quality_score,judge_process_quality_reasoning,judge_context_score,judge_variables_score,judge_relationship_score,judge_hypothesis_reasoning
0,aws-gov.gpt-5.4,L1_load_ocean11,Load the ocean_11_datasets.db database.,1,Data Access,True,1.000,3.5,2285,load_dsi_tool,...,True,True,0.240,[aws-gov.gpt-5.4] The optimal path was a singl...,NaN,NaN,NaN,NaN,NaN,NaN
1,aws-gov.gpt-5.4,L1_load_nif_direct,Load the nif.db database.,1,Data Access,True,1.000,2.9,1914,load_dsi_tool,...,True,True,0.250,[aws-gov.gpt-5.4] The optimal path was a singl...,NaN,NaN,NaN,NaN,NaN,NaN
2,aws-gov.gpt-5.4,L1_load_poisson,Load the poisson.db database.,1,Data Access,True,1.000,2.1,2016,load_dsi_tool,...,True,True,0.250,[aws-gov.gpt-5.4] The reference required a sin...,NaN,NaN,NaN,NaN,NaN,NaN
3,aws-gov.gpt-5.4,L1_search_implosion,Do you have any implosion data in oceans 11 da...,1,Data Discovery,True,1.000,3.2,2310,query_dsi_tool,...,True,True,0.610,[aws-gov.gpt-5.4] The agent completed the task...,NaN,NaN,NaN,NaN,NaN,NaN
4,aws-gov.gpt-5.4,L1_download_nif_sim,Download the simulation data for sim name: nif...,1,Data Retrieval,True,1.000,9.4,2667,"query_dsi_tool, query_dsi_tool, load_dsi_tool,...",...,True,True,0.340,[aws-gov.gpt-5.4] The agent found the correct ...,NaN,NaN,NaN,NaN,NaN,NaN
5,aws-gov.gpt-5.4,L1_store_poisson_hdf5,Download this simulation data file: https://oc...,1,Data Retrieval,True,1.000,2.7,1925,download_file_tool,...,True,True,0.470,[aws-gov.gpt-5.4] The agent completed the down...,NaN,NaN,NaN,NaN,NaN,NaN
6,aws-gov.gpt-5.4,L1_owner_field,"Who is the owner of the NIF dataset""",1,Metadata Query,False,0.857,2.7,1802,query_dsi_tool,...,True,True,0.940,[aws-gov.gpt-5.4] The agent used the same esse...,NaN,NaN,NaN,NaN,NaN,NaN
7,aws-gov.gpt-5.4,L1_extract_dataset_count,How many datasets are in the Ocean 11 catalog?,1,Data Exploration,True,1.000,2.4,1740,query_dsi_tool,...,True,True,1.000,[aws-gov.gpt-5.4] The agent effectively matche...,NaN,NaN,NaN,NaN,NaN,NaN
8,aws-gov.gpt-5.4,L1_extract_originating_org,What organization is listed for the Ocean 11 d...,1,Metadata Query,False,0.857,4.5,1787,query_dsi_tool,...,True,True,0.750,[aws-gov.gpt-5.4] The agent used a single quer...,NaN,NaN,NaN,NaN,NaN,NaN
9,aws-gov.gpt-5.4,L1_extract_physics_theme_count,How many datasets are categorized as physics?,1,Metadata Query,True,1.000,3.3,1745,query_dsi_tool,...,True,True,1.000,[aws-gov.gpt-5.4] The agent executed the same ...,NaN,NaN,NaN,NaN,NaN,NaN


Saved: /Users/hhn/lanl/dsi/tools/ai_search/levels_benchmark_results/20260826_102645_all_levels_summary.json


PosixPath('/Users/hhn/lanl/dsi/tools/ai_search/levels_benchmark_results/20260826_102645_all_levels_summary.json')

## Single-Task Runner

This cell debugs individual tasks or add custom queries.

In [ ]:
def debug_task(task_id: str, model_name: Optional[str] = None):
    """Run and score a single task by ID for interactive debugging.

    Looks up the task in ALL_TASKS, creates a fresh explorer, and prints pass/score.
    Pass model_name (any entry from POSSIBLE_MODELS) to debug that task against a
    specific model instead of the global default LLM_MODEL.
    """
    task = next(t for t in ALL_TASKS if t.task_id == task_id)
    explorer = make_explorer(llm=make_llm(model_name) if model_name else None)
    print(f"Agent workspace: {explorer.wrsk_path}")
    try:
        tr = run_task(explorer, task)
        tr = score_task(task, tr)
    except TaskTimeoutError as e:
        print(f"[TIMEOUT] {e}")
        tr = TaskResult(
            task_id=task.task_id, level=task.level, category=task.category, query=task.query,
            response="", tool_calls=[], elapsed_sec=task.timeout_sec or TASK_TIMEOUT_SEC, tokens=0,
            passed=False, score=0.0, notes=f"timeout: {e}",
        )
    tr.model = model_name or LLM_MODEL
    print(f"Model: {tr.model}")
    print(f"Passed: {tr.passed} | Score: {tr.score}")
    print(f"Tools: {tr.tool_calls}")
    print(f"DSI API calls:\n{format_dsi_call_log(tr.dsi_calls)}")
    if tr.notes:
        print(f"Notes: {tr.notes}")
    if tr.judge_scores:
        print(json.dumps(tr.judge_scores, indent=2))
    return tr

# Example: debug_task("L1_load_ocean11", model_name="anthropic.claude-sonnet-4-5-20250929-v1:0")


In [ ]:
# Run task 1 and display its agent logs
task = LEVEL1_TASKS[0]
explorer = make_explorer()
tr = run_task(explorer, task)
tr = score_task(task, tr)

print(f"Passed: {tr.passed} | Score: {tr.score}")
print(f"DSI API calls:\n{format_dsi_call_log(tr.dsi_calls)}")
print(Path(explorer.wrsk_path, "run.log").read_text())


Creating workspace at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_agent__2026_08_26__10_43


### `L1_load_ocean11` — Level 1 / Data Access

**Query:** Load the ocean_11_datasets.db database.

Loaded `/Users/hhn/lanl/dsi/tools/ai_search/data/oceans_11/ocean_11_datasets.db`.

Tools called: ['load_dsi_tool'] | 2.6s | 2284 tokens
DSI API calls: ['__init__', 'list', 'close', '__init__', 'list', 'close', '__init__', 'list', 'schema', 'close']
Passed: True | Score: 1.0
DSI API calls:
[1] DSI.__init__('/Users/hhn/lanl/dsi/tools/ai_search/data/oceans_11/ocean_11_datasets.db', check_same_thread=False)  [0.001s]
[2] DSI.list(True)  [0.000s]
[3] DSI.close()  [0.000s]
[4] DSI.__init__('/Users/hhn/lanl/dsi/tools/ai_search/data/oceans_11/ocean_11_datasets.db', check_same_thread=False)  [0.001s]
[5] DSI.list(True)  [0.000s]
[6] DSI.close()  [0.000s]
[7] DSI.__init__('/Users/hhn/lanl/dsi/tools/ai_search/data/oceans_11/ocean_11_datasets.db', check_same_thread=False)  [0.000s]
[8] DSI.list(True)  [0.000s]
[9] DSI.schema()  [0.000s]
[10] DSI.close()  [0.000s]
2026-08-26 10:43:19,489 [INFO] global_logger: !!!!!
!!!!!Starting DSIExplorer agent initialization.
2026-08-26 10:43:19,489 [INFO] global_logger: Workspace created at: /Users/hhn/lanl/dsi/tools/ai_search/dsi_explorer_ag

## MCP Mode

To benchmark via MCP (atomic tools on `levels_mcp_server.py`):

1. In a terminal: `python levels_mcp_server.py`
2. Set `USE_MCP = True` in the config cell and re-run setup.

Inspector: `npx @modelcontextprotocol/inspector` → Streamable HTTP → `http://127.0.0.1:8000/mcp`
